# [프로젝트] 시계열 복습

_이 노트북은 LMS에서 내보냈습니다. 영상·퀴즈는 학습 참고용으로 마크다운으로 변환되었습니다._

## 다음 주 운영 예측과 계획표

<div style="background:#FFFFFF;color:#000000;padding:40px;border-radius:12px;border:1px solid #E6E6E6;border-left:4px solid #051C2C">
<div style="font-size:0.9em;color:#656565">시계열 예측 · 여섯째 날 · 모듈 복습</div>
<h1 style="color:#000000;margin:10px 0">시계열 복습</h1>
<div style="font-size:1.05em;color:#656565">1～5일차 결정 규칙과 이상 탐지를 퀴즈 10개로 다시 확인하고, 예측 원점 5개로 고른 운영 예측과 이상 탐지 결과를 계획표 한 장에 담습니다</div>
</div>

**오늘의 질문** 다음 주 168시간의 운영 예측에 어느 모형을 쓰고, 그 결과를 계획표 한 장으로 어떻게 낼까요?
미션 1 은 모듈의 두 모형을 예측 원점 5개로 다시 채점해 운영 예측을 고릅니다. 미션 2 는 그 예측에 `이상` 열을 더해 다음 주 계획표를 만듭니다.

미션에 앞서 퀴즈 10개로 1～5일차 결정 규칙과 이상 탐지 규칙을 서울 공유자전거 출력에 적용합니다. 같은 날의 레퍼런스 레슨을 옆 탭에 열어 두고 풉니다. 문항마다 근거가 그날 카드에 있습니다.

1. 맨 위 설치 셀을 한 번 실행합니다. 런타임 유형은 CPU 로 둡니다. `prophet` 설치에 1～2분 걸리고, 설치 뒤 `cmdstanpy` 오류가 나면 런타임을 다시 시작해 설치 셀부터 다시 실행합니다.
2. 「오늘의 준비」 셀을 실행해 두고 개념 확인 다섯 문항부터 풉니다. 데이터 퀴즈는 그 셀의 출력을 읽고 답합니다.
3. 노트북을 내려받아 Colab 이나 내 컴퓨터에서 열었다면 맨 위의 접힌 `[채점]` 셀을 한 번 ▶ 실행한 뒤 시작합니다.

실습이 처음이라면 세 가지를 먼저 봅니다.

- 화면 위쪽 「Claude 도우미」를 설치해 둡니다(온보딩 레슨 참고). 미션의 PROMPT 를 코드로 바꿔 주는 AI 튜터입니다.
- LMS 에서는 코드 영역 머리에 「채점 준비됨」이 표시된 뒤 시작합니다. 미션 셀을 실행했는데 채점 결과가 나오지 않으면 「채점 준비됨」을 누릅니다.
- 막히면 같은 말을 되풀이하지 말고 PROMPT 에 빠진 조건을 하나 더합니다. 그래도 안 되면 요구를 한 덩어리로 다시 적어 새 메시지로 보냅니다.

In [1]:
# 실습 준비 - 패키지 설치
!pip install -q "numpy>=1.26" "pandas>=2.0" "matplotlib>=3.7" "statsmodels==0.14.6" "prophet==1.3.0"

import os, json, logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.font_manager as fm
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import acf
from statsmodels.stats.diagnostic import acorr_ljungbox
from prophet import Prophet
import warnings; warnings.filterwarnings("ignore")
import cmdstanpy; cmdstanpy.utils.get_logger().disabled = True   # 적합 때마다 찍히는 진행 로그를 숨긴다
logging.getLogger("cmdstanpy").setLevel(logging.CRITICAL)
logging.getLogger("prophet").setLevel(logging.WARNING)

# 한글 폰트 — Colab 이면 나눔고딕을 설치한 뒤 그 ttf 를 폰트 목록에 **직접** 등록한다.
#  설치는 디스크에 파일만 만든다. 이미 떠 있는 폰트 목록은 그 사실을 모르므로 addfont 로
#  등록해야 이름이 보인다(등록하지 않으면 DejaVu Sans 로 밀려 한글이 전부 □ 로 나온다).
import glob, shutil, subprocess
if shutil.which("apt-get"):
    subprocess.run(["apt-get", "-y", "-qq", "install", "fonts-nanum"], check=False)
for _p in glob.glob("/usr/share/fonts/truetype/nanum/Nanum*.ttf"):   # 설치 여부와 무관하게 한 번
    try: fm.fontManager.addfont(_p)
    except Exception: pass
try: fm._get_font.cache_clear()
except Exception: pass
installed = {f.name for f in fm.fontManager.ttflist}
FONT = next((n for n in ["Pretendard", "Apple SD Gothic Neo", "NanumGothic"] if n in installed), "DejaVu Sans")
plt.rcParams["font.family"] = FONT
plt.rcParams["axes.unicode_minus"] = FONT in {"Pretendard", "DejaVu Sans"}   # 나머지 폰트에는 U+2212 가 없다
%config InlineBackend.print_figure_kwargs = {"bbox_inches": None}   # 그림마다 폭이 깎이지 않게
plt.rcParams["figure.dpi"] = plt.rcParams["savefig.dpi"] = 150       # 폭 FIG_W x 150dpi = 1500px 고정
plt.rcParams["figure.autolayout"] = True
plt.rcParams.update({"font.size": 11, "axes.titlesize": 15, "axes.labelsize": 12,
                     "legend.fontsize": 11, "figure.titlesize": 17, "font.weight": "regular",
                     "text.color": "black", "axes.labelcolor": "black", "axes.titlecolor": "black",
                     "axes.edgecolor": "#757575", "xtick.color": "#757575", "ytick.color": "#757575",
                     "xtick.labelcolor": "black", "ytick.labelcolor": "black",
                     "axes.spines.top": False, "axes.spines.right": False, "axes.grid": False})

# 오늘 그림 전체가 함께 쓰는 색 — 베이스라인=남색, 관측=파랑, 강조=하늘색, 문맥=회색
COL = {"남색": "#051C2C", "파랑": "#2251FF", "하늘색": "#00A9F4",
       "회색": "#757575", "연회색": "#B3B3B3"}
FIG_W = 10.0     # 캔버스 폭은 오늘 그림 모두 이 값으로 고정하고 높이만 그림마다 조절한다


def datefmt(ax, fmt="%m-%d", n=5):
    """시간 축 눈금 개수와 표기를 잡아 날짜 라벨이 겹치지 않게 한다"""
    ax.xaxis.set_major_locator(mdates.AutoDateLocator(maxticks=n))
    ax.xaxis.set_major_formatter(mdates.DateFormatter(fmt))


def rmse(a, b):
    """두 계열의 평균 제곱근 오차 — ML 수업에서 쓰던 그 지표 그대로"""
    return float(np.sqrt(np.mean((np.asarray(a, dtype=float) - np.asarray(b, dtype=float)) ** 2)))

print("사용 폰트:", FONT, "| 색 팔레트 준비 완료 — 채점 셀은 LMS 가 자동으로 준비합니다")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 39.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 58.8 MB/s eta 0:00:00
사용 폰트: NanumGothic | 색 팔레트 준비 완료 — 채점 셀은 LMS 가 자동으로 준비합니다


In [ ]:
#@title [채점] 이 셀을 먼저 실행하세요 (한 번이면 됩니다)
#   미션의 「PROMPT 셀」을 실행하면 그 셀의 출력과 그림을 읽어 바로 아래에 채점 결과를 인쇄합니다.
#   통과하면 그 자리에서 완성본 풀이가 열리고, 아니면 항목별 힌트와 튜터에 보낼 질문이 나옵니다.
#   읽지 않고 실행만 해도 됩니다 — 기대값과 풀이는 이 셀에만 있고 「PROMPT 셀」에는 없습니다.
import ast
import base64
import contextlib
import glob
import io
import json
import re
import shutil
import subprocess
import sys

import matplotlib.font_manager as fm      # 아래 폰트 자급이 쓴다
import matplotlib.pyplot as plt
import numpy as np


def _font_bootstrap():
    """한글 폰트 자급 — 맨 위 설치·폰트 셀을 건너뛰어도 그래프의 한글이 깨지지 않게.

    LMS 는 이 셀만 자동 실행하므로 폰트 준비도 여기서 한 번 더 한다. 설치·폰트 셀과 같은
    규칙이라 둘 다 실행돼도 결과가 같고, apt-get 이 없는 자리(로컬)에서는 곧바로 지나간다.
    설치한 ttf 는 `addfont` 로 폰트 목록에 직접 등록한다 — 등록하지 않으면 목록이 옛 상태라
    한글이 □ 로 깨진다(2026-09-09 Colab 실사고).
    """
    try:
        cands = ["Pretendard", "Apple SD Gothic Neo", "NanumGothic"]
        installed = {f.name for f in fm.fontManager.ttflist}
        if not any(n in installed for n in cands) and shutil.which("apt-get"):
            subprocess.run(["apt-get", "-y", "-qq", "install", "fonts-nanum"],
                           check=False, timeout=120)
        # 설치는 파일만 만든다 — 이미 떠 있는 폰트 목록에 addfont 로 등록해야 이름이 보인다.
        # (설치를 건너뛴 자리도 나눔 ttf 가 이미 있으면 이 행들만으로 잡힌다.)
        for _p in glob.glob("/usr/share/fonts/truetype/nanum/Nanum*.ttf"):
            try: fm.fontManager.addfont(_p)
            except Exception: pass
        try: fm._get_font.cache_clear()
        except Exception: pass
        installed = {f.name for f in fm.fontManager.ttflist}
        _font = next((n for n in cands if n in installed), "DejaVu Sans")
        plt.rcParams["font.family"] = _font
        plt.rcParams["axes.unicode_minus"] = _font in {"Pretendard", "DejaVu Sans"}
    except Exception:
        print("한글 폰트를 못 찾아 그래프 글자가 깨질 수 있습니다 —"
              " 맨 위 설치·폰트 셀을 실행하세요")


_font_bootstrap()


def _tail_numbers(tails):
    """라벨 뒤 꼬리에서 괄호 설명을 걷어내고 첫 숫자 하나씩만 모읍니다.

    꼬리 전체의 숫자를 담으면 괄호 안 설명에 우연히 든 수가 기준 범위에 들어 틀린 값을
    통과시킵니다. 예를 들어 「… = 0.705813 (학습 1,848시간 · 주기 24)」 는 기준이 24 부근인
    항목에서 괄호 안의 24 로 열립니다. 그래서 괄호 안은 버리고 맨 앞 숫자 하나만 봅니다.
    """
    picked = []
    for one in tails:
        plain, depth = [], 0
        for ch in one:
            if ch in "([":
                depth += 1
            elif ch in ")]":
                depth = max(0, depth - 1)
            elif depth == 0:
                plain.append(ch)
        got = _nums("".join(plain))
        if got:
            picked.append(got[0])
    return picked


def _dec_json(s):                          # base64(JSON) → 값. 리스트는 원래 튜플이라 되돌립니다
    def _t(v):
        return ({k: _t(x) for k, x in v.items()} if isinstance(v, dict)
                else tuple(_t(x) for x in v) if isinstance(v, list) else v)
    return _t(json.loads(base64.b64decode(s).decode("utf-8")))


# 채점 기준 — base64(JSON), 통과 판정에만 쓴다
_ANS = _dec_json(
           "eyJtMSI6eyJ2YWx1ZXMiOnsi7LGE7YOdIOyYiOy4oeydmCDsm5DsoJAgNeqwnCDt"
           "j4nqt6AgUk1TRSI6WzM2My4wLDM2NC4xLCLrjIAiXSwi67Kg7J207Iqk65287J24"
           "67O064ukIOybkOygkCA16rCcIO2Pieq3oCBSTVNFIOqwgCDsnpHsnYAg7ZuE67O0"
           "IOyImCI6Wy0wLjUsMC41LCLqsJwiXX0sInBsb3QiOnsia2luZCI6ImJhciIsIm4i"
           "OjMsInltYXgiOlszODAuMCwzOTIuMF0sIuy2leudvOuyqCI6dHJ1ZX19LCJtMiI6"
           "eyJ2YWx1ZXMiOnsi7J6E6rOE6rCSIDMuNSDsl5DshJwg6riw7KSAIOq1rOqwhCDt"
           "lZjro6gg7Jik7YOQIOyImCI6WzIuNCwyLjQ2LCLqsbQiXSwi7J6E6rOE6rCSIDUg"
           "7JeQ7IScIOq4sOykgCDqtazqsIQg7ZWY66OoIOyYpO2DkCDsiJgiOlswLjczLDAu"
           "NzcsIuqxtCJdfSwicGxvdCI6eyJraW5kIjoibGluZSIsIm5fc2VyaWVzIjoxLCLs"
           "tpXrnbzrsqgiOnRydWUsImFic21heCI6WzQuNiw0LjldLCJhYnNtYXhfbmFtZSI6"
           "Iuq3uOuemO2UhOydmCDqsJXqsbQgWiJ9LCJmbGFnX2NvbCI6eyJmcmFtZSI6InBs"
           "YW4iLCJjb2wiOiLsnbTsg4EiLCJwaWNrIjoiQUxFUlQiLCJzZXRzIjp7IjMuNSI6"
           "WyIyMDE4LTExLTIxIDAwOjAwIiwiMjAxOC0xMS0yMSAwMTowMCJdLCI0IjpbIjIw"
           "MTgtMTEtMjEgMDE6MDAiXSwiNSI6W119fX19")
_REVEAL = {                                # 완성본 풀이 — base64, 통과할 때만 디코드해 인쇄합니다
    "m1": ("8J+UkyDrr7jshZggMSDsmYTshLHrs7guIEFJIOy9lOuTnCDtkZzsi5zshKAg7JWE656Y7JeQIOuTpOyWtOyZgOyVvCDtlaAg7L2U65Oc64qU"
           "IOydtCDsoJXrj4TsnoXri4jri6QKCuKWoCDsnbQg7L2U65Oc66W8IOu2gOuluCDtlITroaztlITtirgg7JiICgogICAg7J2066+4IOunjOuT"
           "pOyWtOyguCDsnojripQgZGYsIE9SSUdJTlMsIENBTkRfTkFNRVMsIEZDLCBybXNlIOulvCDqt7jrjIDroZwg7JOw6rOgLCBTQVJJTUEg7JmA"
           "IFByb3BoZXQg7J2AIOyerOygge2Vqe2VmOyngCDrp4guCiAgICBPUklHSU5TIOydmCDsm5DsoJDrp4jri6QgRkMg7JiI7Lih7J2YIOyduOuN"
           "seyKpOyduCAxNjjsi5zqsIHsl5DshJwg7Iuk7LihIGRmWyJkZW1hbmQiXSDsmYAg6rOE7KCIIOuCmOydtOu4jCAxNjgg7J24IGRmWyJkZW1h"
           "bmQiXS5zaGlmdCgxNjgpLAogICAgQ0FORF9OQU1FUyDrkZAg7ZuE67O07J2YIFJNU0Ug66W8IHJtc2Ug66GcIOq1rO2VmOqzoCwg7JuQ7KCQ"
           "66eI64ukIOyEuCDqsJLqs7wg6rCA7J6lIOyekeydgCDsmIjsuKEg7J2066aE7J2EIO2VnCDtlonslKkg7Lac66Cl7ZW0IOykmC4g7JuQ7KCQ"
           "IDXqsJwg7Y+J6reg7J20CiAgICDqsIDsnqUg7J6R7J2AIOyYiOy4oeydhCDssYTtg53tlbTshJwgIuyxhO2DnSDsmIjsuKHsnZgg7JuQ7KCQ"
           "IDXqsJwg7Y+J6regIFJNU0UgPSDqsJIg64yAIiDsmYAgIuuyoOydtOyKpOudvOyduOuztOuLpCDsm5DsoJAgNeqwnCDtj4nqt6AgUk1TRSDq"
           "sIAg7J6R7J2AIO2bhOuztCDsiJggPQogICAg6rCSIOqwnCIg66W8IO2VnCDtlonslKksIOyxhO2Dne2VnCDsmIjsuKEg7J2066aE6rO8IOyx"
           "hOygkCDsobDqsbTsnYQg7ZWcIO2WiSDstpzroKXtlbQg7KSYLiDshLgg7JiI7Lih7J2YIOybkOygkCA16rCcIO2Pieq3oCBSTVNFIOulvCDr"
           "p4nrjIAgM+qwnOuhnCDqt7jrpqzqs6AKICAgIOqwgOuhnOy2lSDsnbTrpoQg44CM7JiI7Lih44CNLCDshLjroZzstpUg7J2066aEIOOAjOyb"
           "kOygkCA16rCcIO2Pieq3oCBSTVNFICjrjIAp44CN66W8IOuLrOyVhCDspJguCgogICAgcm93cyA9IFtdCiAgICBmb3IgaywgbyBpbiBlbnVt"
           "ZXJhdGUoT1JJR0lOUyk6CiAgICAgICAgd2VlayA9IEZDW0NBTkRfTkFNRVNbMF1dW2tdLmluZGV4ICAgICAgICAgICAgICAgICAgICAgIyDs"
           "m5DsoJAgMeyLnOqwhCDrkqTrtoDthLAgMTY47Iuc6rCBCiAgICAgICAgYWN0ID0gZGYubG9jW3dlZWssICJkZW1hbmQiXQogICAgICAgIHNj"
           "b3JlcyA9IHsi6rOE7KCIIOuCmOydtOu4jCAxNjgiOiBybXNlKGFjdCwgZGZbImRlbWFuZCJdLnNoaWZ0KDE2OCkubG9jW3dlZWtdKX0KICAg"
           "ICAgICBmb3IgbmFtZSBpbiBDQU5EX05BTUVTOgogICAgICAgICAgICBzY29yZXNbbmFtZV0gPSBybXNlKGFjdCwgRkNbbmFtZV1ba10pCiAg"
           "ICAgICAgcm93cy5hcHBlbmQoc2NvcmVzKQogICAgICAgIGJlc3QgPSBtaW4oc2NvcmVzLCBrZXk9c2NvcmVzLmdldCkKICAgICAgICBwcmlu"
           "dChmIntvOiVtLSVkfSDsm5DsoJA6ICIgKyAiIMK3ICIuam9pbihmIntufSB7djouMmZ964yAIiBmb3IgbiwgdiBpbiBzY29yZXMuaXRlbXMo"
           "KSkKICAgICAgICAgICAgICArIGYiIMK3IOqwgOyepSDsnpHsnYAg7JiI7LihIHtiZXN0fSIpCgogICAgdGFibGUgPSBwZC5EYXRhRnJhbWUo"
           "cm93cywgaW5kZXg9W2Yie286JW0tJWR9IiBmb3IgbyBpbiBPUklHSU5TXSkKICAgIG1lYW41ID0gdGFibGUubWVhbigpICAgICAgICAgICAg"
           "ICAgICAgICAgICAgICAgICAgICAgICAgICAjIOybkOygkCA16rCcIO2Pieq3oCBSTVNFCiAgICBjaG9zZW4gPSBtZWFuNS5pZHhtaW4oKQog"
           "ICAgbl9iZXR0ZXIgPSBpbnQoKG1lYW41W0NBTkRfTkFNRVNdIDwgbWVhbjVbIuqzhOygiCDrgpjsnbTruIwgMTY4Il0pLnN1bSgpKQogICAg"
           "cHJpbnQoZiLssYTtg50g7JiI7Lih7J2YIOybkOygkCA16rCcIO2Pieq3oCBSTVNFID0ge21lYW41W2Nob3Nlbl06LjJmfSDrjIAiKQogICAg"
           "cHJpbnQoZiLrsqDsnbTsiqTrnbzsnbjrs7Tri6Qg7JuQ7KCQIDXqsJwg7Y+J6regIFJNU0Ug6rCAIOyekeydgCDtm4Trs7Qg7IiYID0ge25f"
           "YmV0dGVyfSDqsJwiKQogICAgcHJpbnQoZiLrs7Tqs6A6IOyxhO2DnSB7Y2hvc2VufSDCtyDsmIjsuKEg7JuQ7KCQIDXqsJwoMTDsm5QgMTns"
           "nbzvvZ4xMeyblCAxNuydvCDquIjsmpTsnbwgMjPsi5wpIMK3ICIKICAgICAgICAgICLtlZnsirUgOeyblCAx7J28772e7JuQ7KCQIMK3IOyY"
           "iOy4oSDquLDqsIQgMTY47Iuc6rCEIMK3IOyngO2RnCBSTVNFIikKCiAgICBmaWcsIGF4ID0gcGx0LnN1YnBsb3RzKGZpZ3NpemU9KEZJR19X"
           "LCAzLjQpKQogICAgYXguYmFyKG1lYW41LmluZGV4LCBtZWFuNS52YWx1ZXMsIHdpZHRoPTAuNTUsCiAgICAgICAgICAgY29sb3I9W0NPTFsi"
           "64Ko7IOJIl0sIENPTFsi7ZWY64qY7IOJIl0sIENPTFsi7ZWY64qY7IOJIl1dKQogICAgYXguc2V0X3hsYWJlbCgi7JiI7LihIikKICAgIGF4"
           "LnNldF95bGFiZWwoIuybkOygkCA16rCcIO2Pieq3oCBSTVNFICjrjIApIikKICAgIHBsdC5zaG93KCkKCu2VtOyEpAoKLSDsmIjsg4Eg7IWL"
           "IOqwgOyatOuNsCDri7XsnYAg7IS4IOuyiOynuOyeheuLiOuLpC4g7JuQ7KCQIDXqsJwg7Y+J6regIFJNU0Ug64qUIOqzhOygiCDrgpjsnbTr"
           "uIwgMTY4IOydtCAzNjMuNTXrjIDroZwg6rCA7J6lIOyekeqzoCwg64uk7ISv7Ke4IOuCoCBQcm9waGV0IOq4sOuzuCDqtazshLHsnbQg7JW9"
           "IDM4MeuMgCwg7IWL7Ke4IOuCoCBTQVJJTUEg6rCAIOyVvSAzODXrjIDsnoXri4jri6QuIOuRkCDtm4Trs7Qg66qo65GQIOuyoOydtOyKpOud"
           "vOyduOuztOuLpCDsu6TshJwg7ZuE67O0IOyImOuKlCAw6rCc7J206rOgLCDri6TsnYwg7KO8IOyatOyYgSDsmIjsuKHsnYAg6rOE7KCIIOuC"
           "mOydtOu4jCAxNjgg66GcIOuDheuLiOuLpC4KLSDsm5DsoJDrp4jri6QgMeychOqwgCDrsJTrgJ3ri4jri6QuIDEw7JuUIDI27J286rO8IDEx"
           "7JuUIDnsnbwg7JuQ7KCQ7JeQ7ISc64qUIFNBUklNQSDqsIAg7JW9IDM2OOuMgMK37JW9IDMxOOuMgOuhnCDrsqDsnbTsiqTrnbzsnbggNDE4"
           "LjE464yAwrc0MzAuNzLrjIDrs7Tri6Qg7J6R7JWY7Iq164uI64ukLiDrgpjrqLjsp4Ag7IS4IOybkOygkOyXkOyEnOuKlCDrsqDsnbTsiqTr"
           "nbzsnbjsnbQg6rCA7J6lIOyekeyVmOqzoCwgMTHsm5QgMTbsnbwg7JuQ7KCQ7J2AIOuyoOydtOyKpOudvOyduCAyMDUuMjXrjIAsIFNBUklN"
           "QSDslb0gMzg264yA7J6F64uI64ukLiBQcm9waGV0IOydgCAx7JyE7J24IOybkOygkOydtCDsl4bsirXri4jri6QuIO2VnCDso7zrp4wg6rOo"
           "6528IOuztOuptCDqsrDroaDsnbQg65Kk7KeR7Z6I66+A66GcIOyxhO2DneydgCDtj4nqt6DsnLzroZwg7KCV7ZWp64uI64ukLgotIO2dlO2V"
           "nCDsi6TsiJjripQg65GQIOqwgOyngOyeheuLiOuLpC4g7ZWY64KY64qUIOuyoOydtOyKpOudvOyduOydhCDqsoDspp0gMTY47Iuc6rCEIO2V"
           "nCDrsojsl5DshJzrp4wg6rWs7ZW0IDIwNS4yNeuMgOulvCDtm4Trs7TsnZgg7JuQ7KCQIDXqsJwg7Y+J6regIOyYhuyXkCDsoIHripQg6rKD"
           "7J6F64uI64ukLiDrsqDsnbTsiqTrnbzsnbjrj4Qg6rCZ7J2AIOybkOygkCA16rCc7JeQ7IScIOq1rO2VtOyVvCDsobDqsbTsnbQg6rCZ7Iq1"
           "64uI64ukLiDri6Trpbgg7ZWY64KY64qUIOybkOygkCDsi5zqsIEo6riI7JqU7J28IDIz7IucKSDsnpDssrTrpbwg7LGE7KCQ7JeQIOuEo+yW"
           "tCAxNjjsi5zqsIHsnYQg7ZWcIOyLnOqwhCDslrTquIvrgpjqsowg7J6Q66W064qUIOqyg+yeheuLiOuLpC4gYEZDW+ydtOumhF1ba10uaW5k"
           "ZXhgIOulvCDqt7jrjIDroZwg7JOw66m0IOybkOygkCAx7Iuc6rCEIOuSpOu2gO2EsCAxNjjsi5zqsIHsnbQg66ee7Law7KeR64uI64ukLgot"
           "IOydtOuyiCDso7zsl5Ag7ZWgIOydvDog64uk7J2MIOyjvCDsmIjsuKHsnYAgYGRmWyJkZW1hbmQiXS5zaGlmdCgxNjgpYCDroZwg64OF64uI"
           "64ukLiDtj4nqsIDtkZzsl5DripQg7JuQ7KCQ67OEIFJNU0Ug64uk7ISvIO2WieqzvCDtj4nqt6AsIOyhsOqxtCDtlZwg7ZaJKOyYiOy4oSDs"
           "m5DsoJAgNeqwnCDCtyDtlZnsirUgOeyblCAx7J28772e7JuQ7KCQIMK3IOyYiOy4oSDquLDqsIQgMTY47Iuc6rCEIMK3IOyngO2RnCBSTVNF"
           "KeydhCDtlajqu5gg7KCB7Iq164uI64ukLiDri6TshK/sp7gg64KgIOyhsOqxtOu2gCDqs4TsoIjshLHsspjrn7wg7IOIIOq1rOyEseydhCDs"
           "i5ztl5jtlaAg65WM64+EIOqwmeydgCDsm5DsoJAgNeqwnOuhnCDssYTsoJDtlbQg7ZGc7JeQIO2VnCDsl7TslKkg642U7ZWp64uI64ukLgoK"
           "7L2U65Oc6rCAIOuLteydhCDrp4zrk5zripQg67Cp7IudLiB3ZWVrIOulvCDrp4zrk5zripQg7L2U65OcIO2VnCDtlonsnbQg7JuQ7KCQ66eI"
           "64ukIOyxhOygkO2VoCAxNjjsi5zqsIHsnYQgRkMg7JiI7Lih7J2YIOyduOuNseyKpOyXkOyEnCDqsIDsoLjsmYAsIOyLpOy4ocK367Kg7J20"
           "7Iqk65287J24wrftm4Trs7Qg65GY7J2EIOqwmeydgCDsi5zqsIHsl5DshJwg67mE6rWQ7ZWp64uI64ukLiB0YWJsZS5tZWFuKCkg7J20IOyb"
           "kOygkCA16rCc66W8IO2Pieq3oO2VmOqzoCwgaWR4bWluIOydtCDtj4nqt6DsnbQg6rCA7J6lIOyekeydgCDsmIjsuKHsnYQg6rOg66aF64uI"
           "64ukLiDtm4Trs7Qg7IiY64qUIOuyoOydtOyKpOudvOyduCDsl7TsnYQg67m86rOgIOuRkCDtm4Trs7Trp4wg67mE6rWQ7ZW0IOyFieuLiOuL"
           "pC4K"),
    "m2": ("8J+UkyDrr7jshZggMiDsmYTshLHrs7guIEFJIOy9lOuTnCDtkZzsi5zshKAg7JWE656Y7JeQIOuTpOyWtOyZgOyVvCDtlaAg7L2U65Oc64qU"
           "IOydtCDsoJXrj4TsnoXri4jri6QKCuKWoCDsnbQg7L2U65Oc66W8IOu2gOuluCDtlITroaztlITtirgg7JiICgogICAg66+47IWYIDIg7KSA"
           "67mEIOyFgOydtCDrp4zrk6AgcGxhbiwgQ05ULCBERU0sIFJFU0lELCBIT1VSLCBMQUJFTCwgSU5fQkFTRSwgTUVEX0gsIE1BRF9ILCBBTEVS"
           "VFMg66W8IOq3uOuMgOuhnCDsk7Dqs6Ag64uk7IucIOunjOuTpOyngCDrp4guCiAgICBERU0uc2hpZnQoMTY4KSDsl5DshJwgcGxhbiDsnZgg"
           "MTY47Iuc6rCB7J2EIOqzqOudvCBwbGFuIOydmCAi7JiI7LihIiDsl7TsnYQg66eM65Ok7Ja0IOykmC4KICAgIElOX0JBU0Ug6rCAIOywuOyd"
           "tOqzoCBMQUJFTCDsnbQg6rGw7KeT7J206rOgIFJFU0lEIOqwgCDsnojripQg7Iuc6rCB66eMIOqzqOudvCAwLjY3NDUgw5cgKFJFU0lEIOKI"
           "kiDqt7gg7Iuc6rCB7J2YIE1FRF9IKSDDtyDqt7gg7Iuc6rCB7J2YIE1BRF9IIOuhnCDqsJXqsbQgWiDrpbwg6rOE7IKw7ZW0IOykmC4KICAg"
           "IEFMRVJUUyDsnZgg7J6E6rOE6rCS66eI64ukIOqwleqxtCBaIOygiOuMk+qwkuydtCDsnoTqs4TqsJLsnYQg64SY7J2AIOyLnOqwgSDsiJjr"
           "pbwgSU5fQkFTRSDsnZgg7Iuc6rCEIOyImCDDtyAyNCDroZwg64KY64igICLsnoTqs4TqsJIgMy41IOyXkOyEnCDquLDspIAg6rWs6rCEIO2V"
           "mOujqCDsmKTtg5Ag7IiYID0g6rCSIOqxtCIg7ZiV7Iud7Jy866GcIOyGjOyImCDrkZjsp7gg7J6Q66as6rmM7KeAIO2VnCDtlonslKkg7Lac"
           "66Cl7ZW0IOykmC4KICAgIO2VmOujqCDsmKTtg5Ag7IiYIOyDge2VnOydgCAx6rG07J207JW8LiDqt7gg7IOB7ZWc7J2EIOyxhOyasOuKlCDq"
           "sIDsnqUg7J6R7J2AIOyehOqzhOqwkuycvOuhnCBwbGFuIOydmCAxNjjsi5zqsIHsl5DshJwgQ05UIOyXkOyEnCAi7JiI7LihIiDsnYQg67qA"
           "IOyelOywqOydmCDqsJXqsbQgWiDrpbwg6rCZ7J2AIOuwqeuyleycvOuhnCDqs4TsgrDtlZjqs6AsIOygiOuMk+qwkuydtCDqt7gg7J6E6rOE"
           "6rCS7J2EIOuEmOycvOuptCDssLjsnbggIuydtOyDgSIg7Je07J2EIOunjOuTpOyWtCDspJguCiAgICDqs6Drpbgg7J6E6rOE6rCS6rO8IOyd"
           "tOyDgeycvOuhnCDtg5Dsp4DrkJwg7Iuc6rCBIOyImOulvCDtlZwg7ZaJIOy2nOugpe2VmOqzoCwgMTY47Iuc6rCB7J2YIOqwleqxtCBaIOyE"
           "oCDtlZjrgpjsmYAgwrEg7J6E6rOE6rCSIOqwgOuhnOyEoChheGhsaW5lKeydhCDqt7jroKQg6rCA66Gc7LaVIOOAjOuLpOydjCDso7wg7Iuc"
           "6rCB44CNLCDshLjroZzstpUg44CM7J6U7LCoIOqwleqxtCBa44CNIOydtOumhOydhCDri6zslYQg7KSYLgoKICAgIHBsYW5bIuyYiOy4oSJd"
           "ID0gREVNLnNoaWZ0KDE2OCkubG9jW3BsYW4uaW5kZXhdLnRvX251bXB5KCkgICAgICAgICAgIyDqs4TsoIgg64KY7J2067iMIDE2OAogICAg"
           "YmFzZSA9IHBkLkRhdGFGcmFtZSh7InIiOiBSRVNJRCwgImgiOiBIT1VSfSlbSU5fQkFTRSAmIH5MQUJFTCAmIFJFU0lELm5vdG5hKCldCiAg"
           "ICB6YiA9IDAuNjc0NSAqIChiYXNlWyJyIl0gLSBiYXNlWyJoIl0ubWFwKE1FRF9IKSkgLyBiYXNlWyJoIl0ubWFwKE1BRF9IKQogICAgZGF5"
           "cyA9IGludChJTl9CQVNFLnN1bSgpKSAvIDI0ICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIyDquLDspIAg6rWs6rCEIOuC"
           "oOynnCDsiJggMjgKICAgIHBlcl9kYXkgPSB7YTogaW50KCh6Yi5hYnMoKSA+IGEpLnN1bSgpKSAvIGRheXMgZm9yIGEgaW4gQUxFUlRTfQog"
           "ICAgZm9yIGEgaW4gQUxFUlRTOgogICAgICAgIHByaW50KGYi7J6E6rOE6rCSIHthfSDsl5DshJwg6riw7KSAIOq1rOqwhCDtlZjro6gg7Jik"
           "7YOQIOyImCA9IHtwZXJfZGF5W2FdOi4yZn0g6rG0IikKICAgIEFMRVJUID0gbWluKGEgZm9yIGEgaW4gQUxFUlRTIGlmIHBlcl9kYXlbYV0g"
           "PD0gMSkgICAgICAgICAgICAgICAgICMg7IOB7ZWcIDHqsbTsnYQg7LGE7Jqw64qUIOqwgOyepSDsnpHsnYAg7J6E6rOE6rCSCiAgICBociA9"
           "IHBkLlNlcmllcyhwbGFuLmluZGV4LmhvdXIsIGluZGV4PXBsYW4uaW5kZXgpCiAgICB6ID0gMC42NzQ1ICogKChDTlQubG9jW3BsYW4uaW5k"
           "ZXhdIC0gcGxhblsi7JiI7LihIl0pIC0gaHIubWFwKE1FRF9IKSkgLyBoci5tYXAoTUFEX0gpCiAgICBwbGFuWyLsnbTsg4EiXSA9IHouYWJz"
           "KCkgPiBBTEVSVAogICAgcHJpbnQoZiLqs6Drpbgg7J6E6rOE6rCSIHtBTEVSVH0gwrcg64uk7J2MIOyjvCAxNjjsi5zqsIEg6rCA7Jq0642w"
           "IOydtOyDgeycvOuhnCDtg5Dsp4DrkJwg7Iuc6rCBIHtpbnQocGxhblsn7J207IOBJ10uc3VtKCkpfeqwnCIpCgogICAgZmlnLCBheCA9IHBs"
           "dC5zdWJwbG90cyhmaWdzaXplPShGSUdfVywgMy40KSkKICAgIGF4LnBsb3QocGxhbi5pbmRleCwgeiwgY29sb3I9Q09MWyLtjIzrnpEiXSwg"
           "bHc9MS4yKQogICAgYXguYXhobGluZShBTEVSVCwgY29sb3I9Q09MWyLtmozsg4kiXSwgbHc9MS4wKQogICAgYXguYXhobGluZSgtQUxFUlQs"
           "IGNvbG9yPUNPTFsi7ZqM7IOJIl0sIGx3PTEuMCkKICAgIGF4LnNldF94bGFiZWwoIuuLpOydjCDso7wg7Iuc6rCBIikKICAgIGF4LnNldF95"
           "bGFiZWwoIuyelOywqCDqsJXqsbQgWiIpCiAgICBkYXRlZm10KGF4LCAiJW0tJWQiLCA4KQogICAgcGx0LnNob3coKQoK7ZW07ISkCgotIOyY"
           "iOyDgSDshYsg6rCA7Jq0642wIOuLteydgCDshLgg67KI7Ke4KDUp7J6F64uI64ukLiDquLDspIAg6rWs6rCE7J2YIOudvOuyqCDsl4bripQg"
           "NTk57Iuc6rCB7JeQ7IScIOqwleqxtCBaIOygiOuMk+qwkuydtCAzLjXCtzTCtzUg66W8IOuEmOydgCDsi5zqsIHsnYAgNjjCtzQ2wrcyMeqw"
           "nOyeheuLiOuLpC4g6riw7KSAIOq1rOqwhCAyOOydvOuhnCDrgpjriITrqbQg7ZWY66OoIOyYpO2DkCDsiJjripQgMi40M8K3MS42NMK3MC43"
           "NeqxtOydtOudvCwgMeqxtCDsnbTtlZjrpbwg7LGE7Jqw64qUIOqwgOyepSDsnpHsnYAg7J6E6rOE6rCS7J2AIDUg7J6F64uI64ukLiDsg4Ht"
           "lZzsnYQgMuqxtOycvOuhnCDrkZDrqbQgNCwgM+qxtOycvOuhnCDrkZDrqbQgMy41IOulvCDqs6DrpoXri4jri6QuCi0g64uk7J2MIOyjvOyX"
           "kCDsnbTsg4HsnLzroZwg7YOQ7KeA65CY64qUIOyLnOqwgeydgCDqs6Drpbgg7J6E6rOE6rCS7JeQIOuUsOudvCDri6zrnbzsp5Hri4jri6Qu"
           "IDMuNSDsnbTrqbQgMTHsm5QgMjHsnbwgMOyLnMK3MeyLnCDrkZAg7Iuc6rCB7J206rOgIOqwleqxtCBaIOuKlCDiiJIzLjc0wrfiiJI0Ljc0"
           "IOyeheuLiOuLpC4g65GQIOyLnOqwgeydmCDsi6TsuKHsnYAgMTE564yAwrcxNTnrjIAsIOyYiOy4oeydgCA2MTLrjIDCtzUxN+uMgOyYgOyK"
           "teuLiOuLpC4gNCDsnbTrqbQgMeyLnCDtlZwg7Iuc6rCB7J206rOgLCA1IOydtOuptCAxNjjsi5zqsIEg6rCV6rG0IFog7KCI64yT6rCS7J2Y"
           "IOy1nOuMk+qwkuydtCA0Ljc0IOudvCDsnbTsg4HsnLzroZwg7YOQ7KeA65CY64qUIOyLnOqwgeydtCDsl4bsirXri4jri6QuCi0g7Z2U7ZWc"
           "IOyLpOyImOuKlCDshLgg6rCA7KeA7J6F64uI64ukLiDtlZjrgpjripQg64SY7J2AIOyLnOqwgSDsiJjrpbwg652867Ko7J2EIOu6gCA1OTns"
           "i5zqsIHsnbTrgpgg6rOE7ZqN7ZGcIDE2OOyLnOqwgeycvOuhnCDrgpjriITripQg6rKD7J6F64uI64ukLiDtlZjro6gg7Jik7YOQIOyImOuK"
           "lCDquLDspIAg6rWs6rCE7J2YIOuCoOynnCDsiJggMjgg66GcIOuCmOuIleuLiOuLpC4g64uk66W4IO2VmOuCmOuKlCDqs4Ttmo3tkZwgMTY4"
           "7Iuc6rCB7JeQ7IScIOydtOyDgeycvOuhnCDtg5Dsp4DrkJwg7Iuc6rCBIOyImOulvCDrs7Tqs6Ag7J6E6rOE6rCS7J2EIOqzoOultOuKlCDq"
           "soPsnoXri4jri6QuIOq3uOufrOuptCDri6TsnYwg7KO8IOyLpOy4oeycvOuhnCDquLDspIDsnYQg7KCV7ZWY64qUIOyFiOyeheuLiOuLpC4g"
           "66eI7KeA66eJ7J2AIOqwleqxtCBaIOulvCDrtoDtmLgg6re464yA66GcIOyehOqzhOqwkuqzvCDruYTqtZDtlbQsIOyVhOuemOuhnCDtgazq"
           "sowg67KX7Ja064KcIOyLnOqwgeydhCDrhpPsuZjripQg6rKD7J6F64uI64ukLgotIO2VnOqzhCDrkZAg6rCA7KeA66W8IOqzhO2aje2RnOyZ"
           "gCDtlajqu5gg7KCB7Iq164uI64ukLiBg7J207IOBYCDsl7TsnYAg6re4IOyLnOqwgeydmCDsi6TsuKHsnbQg65Ok7Ja07JmA7JW8IOqzhOyC"
           "sOuQmOuvgOuhnCwg64uk7J2MIOyjvOulvCDslZ7rkZDqs6Ag67O064K064qUIOqzhO2aje2RnOyXkOyEnOuKlCDruYTslrQg7J6I6rOgIO2V"
           "nCDso7wg64+Z7JWIIOyxhOybjOynkeuLiOuLpC4g7IOB7ZWc7J2EIDHqsbTsnLzroZwg6rOg66W066m0IDEx7JuUIDIx7J28IDDsi5zCtzHs"
           "i5zsspjrn7wg7JiI7Lih67O064ukIDM1MOuMgCDrhJjqsowg66qo7J6Q656AIOyLnOqwgeuPhCDsnbTsg4HsnLzroZwg7YOQ7KeA65CY7KeA"
           "IOyViuyKteuLiOuLpC4KLSDsnbTrsogg7KO87JeQIO2VoCDsnbw6IOqzhO2aje2RnOydmCBg7JiI7LihYCDsl7TsnYAg6riI7JqU7J287JeQ"
           "IOyxhOybjCDrs7TrgrTqs6AsIO2RnCDslYTrnpjsl5Ag44CM6riw7KSAIOq1rOqwhCAxMOyblCAyMOydvO+9njEx7JuUIDE27J28IMK3IOye"
           "hOqzhOqwkiA1IMK3IO2VmOujqCDsmKTtg5Ag7IiYIDAuNzXqsbTjgI3sspjrn7wg6rOg66W4IOyehOqzhOqwkuqzvCDqt7zqsbDrpbwg7ZWc"
           "IO2WieycvOuhnCDsoIHsirXri4jri6QuIGDsnbTsg4FgIOyXtOydgCDsi6TsuKHsnbQg65Ok7Ja07Jik64qUIOuMgOuhnCDqsJnsnYAgTUVE"
           "X0jCt01BRF9IIOyZgCDsnoTqs4TqsJLsnLzroZwg7LGE7JuB64uI64ukLgoK7L2U65Oc6rCAIOuLteydhCDrp4zrk5zripQg67Cp7IudLiBi"
           "YXNlIOulvCDrp4zrk5zripQg7L2U65OcIO2VnCDtlonsnbQg6riw7KSAIOq1rOqwhOydmCDrnbzrsqgg7JeG64qUIOyelOywqCA1OTnsi5zq"
           "sIHsnYQg6rOg66W06rOgLCBtYXAg7J20IO2WieuniOuLpCDqt7gg7Iuc6rCB7J2YIE1FRF9IwrdNQURfSCDrpbwg6rCA7KC47Ji164uI64uk"
           "LiBwZXJfZGF5IOuKlCDrhJjsnYAg7Iuc6rCBIOyImOulvCBJTl9CQVNFIOydmCDrgqDsp5wg7IiYIDI4IOuhnCDrgpjriIgg6rCS7J206rOg"
           "LCBtaW4g7J20IOyDge2VnCAx6rG07J2EIOyxhOyasOuKlCDqsIDsnqUg7J6R7J2AIOyehOqzhOqwkuydhCDqs6DrpoXri4jri6QuIOqzhO2a"
           "je2RnCAxNjjsi5zqsIHsnZgg6rCV6rG0IFog64+EIOqwmeydgCDsi5zqsIHrs4Qg7KSR7JWZ6rCS6rO8IE1BRCDroZwg6rOE7IKw7ZWp64uI"
           "64ukLgo="),
}

#  힌트는 (한 문장 힌트, 튜터에게 물을 것) 짝입니다 — 채점이 ❌ 인 항목에만 나옵니다
# 항목별 힌트 — base64(JSON)
_HINT = _dec_json(
           "eyJtMSI6eyLssYTtg50g7JiI7Lih7J2YIOybkOygkCA16rCcIO2Pieq3oCBSTVNF"
           "IjpbIuybkOygkOuniOuLpCBgRkNb7J2066aEXVtrXS5pbmRleGAg7J2YIDE2OOyL"
           "nOqwgeyXkOyEnCDsi6TsuKEgYGRmW1wiZGVtYW5kXCJdYCDsmYAg7IS4IOyYiOy4"
           "oeydhCDruYTqtZDtlojripTsp4Ag67O07IS47JqULiDqs4TsoIgg64KY7J2067iM"
           "IDE2OCDrj4Qg6rCZ7J2AIDE2OOyLnOqwgeydmCBgZGZbXCJkZW1hbmRcIl0uc2hp"
           "ZnQoMTY4KWAg66GcIOq6vOuCtCDtm4Trs7Qg65GY6rO8IO2VqOq7mCDsm5DsoJAg"
           "NeqwnOulvCDtj4nqt6DtlZjqs6AsIO2Pieq3oOydtCDqsIDsnqUg7J6R7J2AIOyY"
           "iOy4oeydmCDqsJLsnYQg44CM7J2066aEID0g7Iir7J6QIOuMgOOAjSDtlZwg7ZaJ"
           "7Jy866GcIOy2nOugpe2VqeuLiOuLpCIsIuydtCDsvZTrk5zqsIAg7JuQ7KCQ66eI"
           "64ukIOyLpOy4oeqzvCDsmIjsuKHsnYQg7Ja064qQIDE2OOyLnOqwgeyXkOyEnCDr"
           "uYTqtZDtlojripTqsIAiXSwi67Kg7J207Iqk65287J2467O064ukIOybkOygkCA1"
           "6rCcIO2Pieq3oCBSTVNFIOqwgCDsnpHsnYAg7ZuE67O0IOyImCI6WyLtm4Trs7Qg"
           "MuqwnOunjCDshLjqs6Ag67Kg7J207Iqk65287J24IOyekOyLoOydgCDrubzripTs"
           "p4AsIOu2gOuTse2YuOqwgCDjgIzrsqDsnbTsiqTrnbzsnbjrs7Tri6Qg7J6R64uk"
           "44CNIOyqveyduOyngCDtmZXsnbjtlZjshLjsmpQuIOOAjOydtOumhCA9IOyIq+ye"
           "kCDqsJzjgI0g7ZWcIO2WieyeheuLiOuLpCIsIuydtCDsvZTrk5zqsIAg7ZuE67O0"
           "IOyImOulvCDshYAg65WMIOustOyXh+qzvCDrrLTsl4fsnYQg67mE6rWQ7ZaI64qU"
           "6rCAIl0sInBsb3QiOlsi66eJ64yA6rCAIDPqsJwo6rOE7KCIIOuCmOydtOu4jCAx"
           "Njgg6rO8IO2bhOuztCAy6rCcKeyduOyngCwg66eJ64yAIOuGkuydtOqwgCDsm5Ds"
           "oJAgNeqwnCDtj4nqt6AgUk1TRSDsnbjsp4AsIOqwgOuhnOy2lcK37IS466Gc7LaV"
           "7JeQIOydtOumhOydtCDsnojripTsp4Ag7LCo66GA66GcIO2ZleyduO2VmOyEuOya"
           "lCIsIuuCtCDqt7jrprzsnZgg66eJ64yAIOqwnOyImOyZgCDrhpLsnbQsIOy2lSDs"
           "nbTrpoTsnbQg6riw7KSA6rO8IOqwmeydgOyngCJdLCJlcnJvciI6WyJOYW1lRXJy"
           "b3Ig66m0IOuwlOuhnCDsnIQg44CM66+47IWYIDEg7KSA67mEIOyFgOOAjeydhCDr"
           "qLzsoIAg7Iuk7ZaJ7ZaI64qU7KeA67aA7YSwIO2ZleyduO2VmOyEuOyalC4g7Iuk"
           "7ZaJ7ZaI64qU642w64+EIOyYpOulmOqwgCDrgpjrqbQg7Jik66WYIOusuOyepSDs"
           "sqsg7ZaJ7J2EIOq3uOuMgOuhnCDtipzthLDsl5Ag67O07JesIOyjvOqzoCwg44CM"
           "66+47IWYIDEg7KSA67mEIOyFgOOAjeydtCDrp4zrk6Ag7J2066aEKGRmIMK3IE9S"
           "SUdJTlMgwrcgQ0FORF9OQU1FUyDCtyBGQyDCtyBybXNlKeydgCDri6Tsi5wg66eM"
           "65Ok7KeAIOunkOqzoCDqt7jrjIDroZwg7JOw65286rOgIOu2hOuqhe2eiCDsoIHs"
           "nLzshLjsmpQiLCLsnbQg7Jik66WY6rCAIOyWtOuKkCDsvZTrk5wg7ZaJ7JeQ7ISc"
           "IOyZnCDrgqzripTsp4AiXX0sIm0yIjp7IuyehOqzhOqwkiAzLjUg7JeQ7IScIOq4"
           "sOykgCDqtazqsIQg7ZWY66OoIOyYpO2DkCDsiJgiOlsi6riw7KSAIOq1rOqwhChg"
           "SU5fQkFTRWAp7JeQ7IScIOudvOuyqChgTEFCRUxgKeydtCDsl4bqs6Ag7J6U7LCo"
           "KGBSRVNJRGAp6rCAIOyeiOuKlCDsi5zqsIHrp4wg6rOo6528IOqwleqxtCBaIOul"
           "vCDqs4TsgrDtlojripTsp4AsIOuEmOydgCDsi5zqsIEg7IiY66W8IOq4sOykgCDq"
           "tazqsITsnZgg64Kg7KecIOyImChgSU5fQkFTRWAg7J2YIOyLnOqwhCDsiJggw7cg"
           "MjQp66GcIOuCmOuItOuKlOyngCDtmZXsnbjtlZjshLjsmpQuIOudvOuyqOydhCDr"
           "uoAg7Iuc6rCEIOyImOuCmCDqs4Ttmo3tkZzsnZggMTY47Iuc6rCB7Jy866GcIOuC"
           "mOuIhOuptCDqsJLsnbQg64us65287KeR64uI64ukLiIsIuydtCDsvZTrk5zqsIAg"
           "6riw7KSAIOq1rOqwhOydmCDslrTripAg7Iuc6rCB7J2EIOqzqOuekOqzoCDrhJjs"
           "nYAg7Iuc6rCBIOyImOulvCDrrLTsl4fsnLzroZwg64KY64i064qU6rCAIl0sIuye"
           "hOqzhOqwkiA1IOyXkOyEnCDquLDspIAg6rWs6rCEIO2VmOujqCDsmKTtg5Ag7IiY"
           "IjpbIuqwleqxtCBaIOydmCDsoIjrjJPqsJLsnYQg7J6E6rOE6rCS6rO8IOu5hOq1"
           "kO2WiOuKlOyngCwg7Iuc6rCB67OEIOykkeyVmeqwkuqzvCBNQUQg66W8IOq3uCDt"
           "lonsnZgg7Iuc6rCBKGBIT1VSYCnsl5Ag66ee7LawIOqwgOyguOyZlOuKlOyngCDt"
           "mZXsnbjtlZjshLjsmpQuIOu2gO2YuCDqt7jrjIDroZwg67mE6rWQ7ZWY66m0IOyV"
           "hOuemOuhnCDtgazqsowg67KX7Ja064KcIOyLnOqwgeydtCDruaDsp4Dqs6AsIOyg"
           "hOyytCDspJHslZnqsJIg7ZWY64KY66GcIOuCmOuIhOuptCDqsJLsnbQg7YGs6rKM"
           "IOuLrOudvOynkeuLiOuLpCIsIuydtCDsvZTrk5zqsIAgTUVEX0gg7JmAIE1BRF9I"
           "IOulvCDslrTripAg7Iuc6rCB7JeQIOunnuy2sCDqsIDsoLjsmZTqs6Ag7J6E6rOE"
           "6rCS6rO8IOyWtOuWu+qyjCDruYTqtZDtlojripTqsIAiXSwicGxvdCI6WyLqs4Tt"
           "mo3tkZwgMTY47Iuc6rCB7J2YIOyelOywqCDqsJXqsbQgWiDshKAg7ZWY64KY7JmA"
           "IOqzoOuluCDsnoTqs4TqsJLsnZgg6rCA66Gc7ISgIOuRkCDspIQoYGF4aGxpbmVg"
           "KeydtCDsnojripTsp4AsIOqwgOuhnOy2leqzvCDshLjroZzstpXsl5Ag7J2066aE"
           "7J20IOyeiOuKlOyngCDtmZXsnbjtlZjshLjsmpQiLCLrgrQg6re466a87JeQIOyE"
           "oOydtCDrqocg6rCcIOq3uOugpOyhjOqzoCDrkZAg7LaV7JeQIOydtOumhOydtCDs"
           "nojripTsp4AiXSwiZXJyb3IiOlsiTmFtZUVycm9yIOuptCDrsJTroZwg7JyEIOOA"
           "jOuvuOyFmCAyIOykgOu5hCDshYDjgI3snYQg66i87KCAIOyLpO2Wie2WiOuKlOyn"
           "gOu2gO2EsCDtmZXsnbjtlZjshLjsmpQuIOyLpO2Wie2WiOuKlOuNsOuPhCDsmKTr"
           "pZjqsIAg64KY66m0IOyYpOulmCDrrLjsnqUg7LKrIO2WieydhCDqt7jrjIDroZwg"
           "7Yqc7YSw7JeQIOuztOyXrCDso7zqs6AsIOOAjOuvuOyFmCAyIOykgOu5hCDshYDj"
           "gI3snbQg66eM65OgIOydtOumhChwbGFuIMK3IENOVCDCtyBERU0gwrcgUkVTSUQg"
           "wrcgSE9VUiDCtyBMQUJFTCDCtyBJTl9CQVNFIMK3IE1FRF9IIMK3IE1BRF9IIMK3"
           "IEFMRVJUUynsnYAg64uk7IucIOunjOuTpOyngCDrp5Dqs6Ag6re464yA66GcIOyT"
           "sOudvOqzoCDrtoTrqoXtnogg7KCB7Jy87IS47JqUIiwi7J20IOyYpOulmOqwgCDs"
           "lrTripAg7L2U65OcIO2WieyXkOyEnCDsmZwg64Ks64qU7KeAIl0sInBsb3RfeiI6"
           "WyLqt7jrnpjtlITsnZgg7ISg7J20IOqzhO2aje2RnCAxNjjsi5zqsIHsnZgg6rCV"
           "6rG0IFog7J247KeAIO2ZleyduO2VmOyEuOyalC4g6rOE7ZqN7ZGcIOqwleqxtCBa"
           "IOuKlCDsi6TsuKEgYENOVGAg7JeQ7IScIGDsmIjsuKFgIOyXtOydhCDruoAg7J6U"
           "7LCo66W8IOyTsOqzoCwgMC42NzQ1IMOXICjsnpTssKgg4oiSIOq3uCDsi5zqsIEg"
           "YE1FRF9IYCkgw7cg6re4IOyLnOqwgSBgTUFEX0hgIOuhnCDqs4TsgrDtlanri4jr"
           "i6QuIOyLpOy4oeydtOuCmCDsnpTssKjrpbwg6re464yA66GcIOq3uOumrOuptCDs"
           "hKDsnZgg7YGs6riw6rCAIOq4sOykgOqzvCDri6zrnbzsp5Hri4jri6QiLCLsnbQg"
           "7L2U65Oc6rCAIOq3uOuemO2UhOyXkCDqt7jrprAg7ISg7J2AIOyWtOuWpCDqsJLs"
           "nbTqs6AsIOq3uCDsnpTssKjripQg66y07JeH7JeQ7IScIOustOyXh+ydhCDruoAg"
           "6rCS7J246rCAIl0sImZsYWciOlsiYOydtOyDgWAg7Je07J2AIOqzhO2aje2RnCDq"
           "sJXqsbQgWiDsnZgg7KCI64yT6rCS7J20IOqzoOuluCDsnoTqs4TqsJIgYEFMRVJU"
           "YCDrpbwg64SY64qUIOyLnOqwgeunjCDssLjsnoXri4jri6QuIOqzhO2aje2RnCDq"
           "sJXqsbQgWiDripQg7Iuk7LihIGBDTlRgIOyXkOyEnCBg7JiI7LihYCDsl7TsnYQg"
           "67qAIOyelOywqOulvCDsk7Dqs6AsIOq3uCDsi5zqsIHsnZggYE1FRF9IYCDrpbwg"
           "67qAIOuSpCBgTUFEX0hgIOuhnCDrgpjriJXri4jri6QuIOu5hOq1kO2VnCDqsJLs"
           "nbQg6re4656Y7ZSE7JeQIOq3uOumsCDqsJXqsbQgWiDsmYAg6rCZ7J2A7KeALCDr"
           "uYTqtZDtlZwg7J6E6rOE6rCS7J20IOqzoOuluCDsnoTqs4TqsJLqs7wg6rCZ7J2A"
           "7KeAIO2ZleyduO2VmOyEuOyalCIsIuydtCDsvZTrk5zqsIAgYOydtOyDgWAg7Je0"
           "7J2EIOyWtOuKkCDqsJLqs7wg7Ja064qQIOyehOqzhOqwkuydhCDruYTqtZDtlbQg"
           "7LGE7Jug64qU6rCAIl0sImVycm9yX2tpbmRzIjpbeyJtYXRjaCI6W1siVmFsdWVF"
           "cnJvciIsImFyZyg/OnVtZW50KT8gaXMgKD86YW4gKT9lbXB0eXx6ZXJvLXNpemUg"
           "YXJyYXkiXSxbIlN0b3BJdGVyYXRpb24iLCIiXV0sImhpbnQiOiLsg4HtlZzsnYQg"
           "7LGE7Jqw64qUIOyehOqzhOqwkuydtCDsl4bsirXri4jri6QuIOyEuCDsnoTqs4Tq"
           "sJIg66qo65GQIO2VmOujqCDsmKTtg5Ag7IiY6rCAIOyDge2VnOydhCDrhJjslrQg"
           "6rOg66W8IOyehOqzhOqwkiDrqqnroZ3snbQg67mE7JeI7Iq164uI64ukLiDquLDs"
           "pIAg6rWs6rCEIOqwleqxtCBaIOulvCDsi5zqsIHrs4QgYE1FRF9IYMK3YE1BRF9I"
           "YCDroZwg6rOE7IKw7ZaI64qU7KeALCDtlZjro6gg7Jik7YOQIOyImOulvCDquLDs"
           "pIAg6rWs6rCEIOuCoOynnCDsiJgoMjgp66GcIOuCmOuItOuKlOyngCDtmZXsnbjt"
           "lZjshLjsmpQiLCJhc2siOiLsnbQg7L2U65Oc6rCAIOq4sOykgCDqtazqsIQg6rCV"
           "6rG0IFog66W8IOyWtOuKkCDspJHslZnqsJLqs7wgTUFEIOuhnCDqs4TsgrDtlojq"
           "s6AsIOuEmOydgCDsi5zqsIEg7IiY66W8IOustOyXh+ycvOuhnCDrgpjriLTripTq"
           "sIAifV19fQ==")

_ASK = {"m1": "full", "m2": "full"}
_ASK_HEAD = {"full": "◼ 튜터에 그대로 보낼 질문 (코드는 고치지 말라고 명시합니다)",
             "blank": "◼ 튜터에 보낼 질문 (빈칸은 내가 채웁니다)",
             "bare": "◼ 튜터에 보낼 질문의 항목 (문장으로 만드는 것은 내가 합니다)",
             "self": "◼ 스스로 점검할 항목 (이 미션은 힌트 없이 내가 명세합니다)"}
_ASK_FORM = {"full": "내 PROMPT 는 「{prompt}」 였는데 「{name}」 채점이 ❌ 였습니다."
                     " AI 코드 표시선 아래 코드를 보고 「{ask}」 에 3~5문장으로 답해 주세요. 코드는 고치지 마세요.",
             "blank": "내 PROMPT 는 ____ 였는데 「{name}」 채점이 ❌ 였습니다. AI 코드 표시선 아래 코드에서"
                      " 「{ask}」 부터 짚고, 어느 행을 무엇으로 고칠지 한 문장으로 답해 주세요.",
             "bare": "무엇을 : {ask} / 어디를 : {name} / 고치지 말 것 : 코드",
             "self": "무엇을 : {ask} / 어디를 : {name} / 다시 볼 곳 : AI 코드 표시선 아래 코드"}

_FIGS = []                                 # 「PROMPT 셀」이 만든 그림을 모으는 자리
_STATE = {"tag": None, "buf": None, "out": None, "ids0": set(), "src": ""}
_TRY = {}                                  # 미션마다 ❌ 가 몇 번째인지 — 결정 16 이 이 수를 봅니다
_orig_figure = getattr(plt.figure, "_orig", plt.figure)   # 이 셀을 다시 실행해도 겹겹이 싸이지 않게


def _figure(*a, **k):
    f = _orig_figure(*a, **k)
    _FIGS.append(f)
    return f


_figure._orig = _orig_figure
plt.figure = _figure                       # subplots · pandas · seaborn 도 이 경로를 거칩니다


def _get(name):
    """이름이 살아 있으면 그 값, 없으면 None — 아직 만들지 않은 이름에도 채점이 죽지 않게"""
    return globals().get(name)


def _open_figs():
    """지금 열려 있는 그림들 — 래퍼를 거치지 않고 읽습니다"""
    try:
        return [plt._pylab_helpers.Gcf.figs[n].canvas.figure for n in plt.get_fignums()]
    except Exception:
        return []


def _nums(text):
    """출력 텍스트에서 숫자만 뽑습니다 (천 단위 쉼표 제거, 지수 표기 포함)"""
    picked = []
    for seg in re.findall(r"-?\d[\d,]*\.?\d*(?:[eE][-+]?\d+)?", text or ""):
        try:
            picked.append(float(seg.replace(",", "")))
        except ValueError:
            continue
    return picked


def _value_lines(txt, units):
    """값 채점이 볼 행만 고릅니다 — 단위가 붙은 행, 또는 「이름 = 값」 꼴의 행.

    표를 통째로 띄우면 그 안에 정답이 우연히 섞여 있어, 「값을 골라 단위와 함께 출력한다」 는
    계약을 지키지 않고도 ✅ 가 열립니다. 그래서 단위를 달아 고른 행이 하나라도 있으면 그
    행들만 보고, 그런 행이 아예 없을 때만 출력 전체를 봅니다.
    """
    picked = []
    for line in (txt or "").splitlines():
        if not line.strip():
            continue
        if "=" in line or any(unit and unit in line for unit in units):
            picked.append(line)
    return picked


def _label_lines(val_lines, label):
    """라벨이 든 값줄만 골라, 그 줄에서 라벨 뒤(「=」·「:」 뒤) 토막을 돌려줍니다.

    값줄의 숫자를 한 주머니에 담아 띠와 대조하면 **옆 칸의 값**이 이 칸을 헛통과시킵니다 —
    부호 칸 둘이 같은 띠일 때 한쪽에 찍힌 1 이 다른 칸까지 열어 주던 자리입니다. 그래서
    라벨이 든 줄이 있으면 그 줄의 라벨 뒤만 봅니다(없으면 부르는 쪽이 값줄 전체로 물러납니다).
    라벨 찾기는 정확히 같은 문자열 → 못 찾으면 공백·괄호를 걷어낸 꼴, 두 단입니다.
    """
    def _tighten(txt):                     # 공백과 괄호(안의 말까지) 를 걷어낸 글자 · 원래 자리
        keep, depth = [], 0
        for i, ch in enumerate(txt or ""):
            if ch in "([":
                depth += 1
            elif ch in ")]":
                depth = max(0, depth - 1)
            elif depth == 0 and not ch.isspace():
                keep.append(i)
        return "".join((txt or "")[i] for i in keep), keep

    tight_label = _tighten(label)[0]
    tails = []
    for line in val_lines:
        end = line.find(label) + len(label) if label and label in line else -1
        if end < 0 and tight_label:                       # 2단 — 공백·괄호를 걷어내고 다시 봅니다
            tight_line, keep = _tighten(line)
            at = tight_line.find(tight_label)
            end = (keep[at + len(tight_label) - 1] + 1) if at >= 0 else -1
        if end < 0:
            continue
        tail = line[end:]
        cut_at = [p for p in (tail.find("="), tail.find(":")) if p >= 0]
        tails.append(tail[min(cut_at) + 1:] if cut_at else tail)
    return tails


def _axes(fig):
    """Figure · Axes 어느 쪽을 받아도 축 목록으로 폅니다"""
    if fig is None:
        return []
    axs = getattr(fig, "axes", None)
    if isinstance(axs, list):
        return axs
    return [fig] if hasattr(fig, "get_xlabel") else []


def _seg_lines(ax):
    """`vlines` · `hlines` 로 그은 선 — Line2D 가 아니라 **LineCollection** 이라 `ax.lines` 에 없습니다.

    토막 하나를 `(가로값, 세로값)` 한 쌍으로 펴서 돌려줍니다. 이 경로를 열어 두지 않으면
    `plt.hlines(...)` 로 화면에 분명히 그어 놓은 구간 막대가 「막대가 하나도 없다」 로 채점돼,
    학생이 고칠 곳 없는 결함을 고치러 가게 됩니다.
    """
    picked = []
    for c in getattr(ax, "collections", []):
        get_segs = getattr(c, "get_segments", None)      # LineCollection 에만 있는 메서드입니다
        if get_segs is None:                              # 산점도(PathCollection)·채움 영역(PolyCollection)은 선이 아닙니다
            continue
        try:
            segs = get_segs()
        except Exception:
            continue
        for seg in segs:
            xy = np.asarray(seg, dtype=float).reshape(-1, 2)
            if xy.shape[0] >= 2:
                picked.append((xy[:, 0], xy[:, 1]))
    return picked


def _lines(ax):
    """축의 선을 곡선(점 10개 이상)과 수직선(x 가 같은 두 점)으로 갈라 돌려줍니다"""
    cur, vline = [], []
    cands = []
    for ln in ax.lines:                     # 선 하나가 깨져도 나머지는 살린다 — `axvline(x=날짜)` 처럼 x 가
        try:                                #  Timestamp 인 세로선(add_changepoints_to_plot)은 날짜를 숫자로 바꿔 받는다
            xd = np.asarray(ln.get_xdata(), dtype=float).reshape(-1)
            yd = np.asarray(ln.get_ydata(), dtype=float).reshape(-1)
        except (TypeError, ValueError):
            try:
                import matplotlib.dates as _mdates
                xd = np.asarray(_mdates.date2num(ln.get_xdata()), dtype=float).reshape(-1)
                yd = np.asarray(ln.get_ydata(), dtype=float).reshape(-1)
            except Exception:
                continue
        cands.append((xd, yd))
    cands += _seg_lines(ax)                     # `plt.vlines` 로 그은 수직선도 여기서 합류합니다
    for xd, yd in cands:
        if xd.size >= 10:
            cur.append((xd, yd))
        elif xd.size == 2 and np.isclose(xd[0], xd[1]):
            vline.append(float(xd[0]))
    return cur, vline


def _marker_lines(ax):
    """마커를 찍은 Line2D — `plot(x, y, "o")` 든 `plot(x, y, marker="o")` 든 점으로 셉니다.

    선으로 이으면서 점도 함께 찍은 선(`linestyle` 이 기본 실선)까지 세는 것은, 점추정을 그렇게
    찍은 그림이 「점추정 마커가 0개다」 로 ❌ 나던 자리이기 때문입니다. 점 계수는 linestyle 과
    무관하고, 마커를 달지 않은 선만 점에서 뺍니다.
    """
    picked = []
    for ln in ax.lines:
        marker = str(ln.get_marker()).strip()
        if marker in ("", "None", "none"):     # 마커 없는 선은 점이 아닙니다 — 선 모양은 보지 않습니다
            continue
        if marker in ("|", "_"):               # errorbar 의 capsize 캡은 점이 아닙니다 — 구간 끝을 그은 짧은 선입니다
            continue
        try:                                # 날짜 x 인 선 하나가 나머지 점까지 죽이지 않게 갈라 받는다
            xd = np.asarray(ln.get_xdata(), dtype=float).reshape(-1)
            yd = np.asarray(ln.get_ydata(), dtype=float).reshape(-1)
        except (TypeError, ValueError):
            try:
                import matplotlib.dates as _mdates
                xd = np.asarray(_mdates.date2num(ln.get_xdata()), dtype=float).reshape(-1)
                yd = np.asarray(ln.get_ydata(), dtype=float).reshape(-1)
            except Exception:
                continue
        if xd.size and xd.size == yd.size:
            picked.append((ln, xd, yd))
    return picked


def _pt_sets(ax):
    """산점도의 점 좌표 — `scatter`(PathCollection)와 `plot(x, y, "o")` 를 함께 셉니다"""
    marker = _marker_lines(ax)
    pt_sets = [np.asarray(c.get_offsets(), dtype=float).reshape(-1, 2)
          for c in ax.collections if not hasattr(c, "get_segments")]   # 선 모음은 점으로 세지 않습니다
    pt_sets += [np.column_stack([xd, yd]) for _, xd, yd in marker]
    return marker, pt_sets


def _spans(ax):
    """가로로 누운 구간 막대를 (왼쪽 끝, 오른쪽 끝) 짝으로 모읍니다.

    `hlines`(LineCollection) · `plot([lo, hi], [y, y])`(Line2D) · `errorbar(xerr=…)` ·
    `barh`(Rectangle) 어느 방법으로 그렸어도 같은 자리에서 셉니다 — 그리는 방법을 하나로
    못 박으면 AI 가 고른 다른 방법이 「막대가 없다」 로 읽혀 학생이 애먼 곳을 고칩니다.
    세로로 선 토막(수직 기준선)은 구간이 아니라 여기서 빠집니다.
    """
    picked = []
    cands = [(np.asarray(ln.get_xdata(), dtype=float).reshape(-1),
           np.asarray(ln.get_ydata(), dtype=float).reshape(-1)) for ln in ax.lines]
    cands += _seg_lines(ax)
    for xd, yd in cands:
        if xd.size != 2 or yd.size != 2:
            continue
        if not (np.isfinite(xd).all() and np.isfinite(yd).all()):
            continue
        if not np.isclose(yd[0], yd[1]) or float(xd.max() - xd.min()) <= 0:
            continue                                   # 가로로 누워 있고 길이가 있는 토막만
        picked.append((float(xd.min()), float(xd.max())))
    for p in getattr(ax, "patches", []):               # `barh(left=lo, width=hi-lo)` 로 그린 구간
        try:
            x0, bar_w, heights = float(p.get_x()), float(p.get_width()), float(p.get_height())
        except (AttributeError, TypeError, ValueError):
            continue
        if bar_w > 0 and heights > 0 and np.isfinite(x0):
            picked.append((x0, x0 + bar_w))
    return picked


def _near(val, ref, factor=1.5):
    """실측값이 기준의 배수 안에 드는가 — 눈금·표본 차이는 넘어갑니다"""
    try:
        val = float(val)
    except (TypeError, ValueError):
        return False
    if ref == 0:
        return abs(val) < 1e-9
    return val * ref > 0 and abs(ref) / factor <= abs(val) <= abs(ref) * factor


def _diagonal(ax, skip=(), xy=None):
    """y = x 인 기준선이 점이 놓인 범위를 가로지르는가 — 선이 있기만 하면 통과가 아닙니다"""
    drop_ids = {id(ln) for ln in skip}
    cands = [(np.asarray(ln.get_xdata(), dtype=float).reshape(-1),
           np.asarray(ln.get_ydata(), dtype=float).reshape(-1))
          for ln in ax.lines if id(ln) not in drop_ids]
    cands += _seg_lines(ax)                     # LineCollection 으로 그은 y = x 기준선도 셉니다
    for xd, yd in cands:
        if xd.size < 2 or xd.size != yd.size:
            continue
        if not (np.isfinite(xd).all() and np.isfinite(yd).all()):
            continue
        width = float(xd.max() - xd.min())
        if width <= 0:                                    # 수직선은 대각선이 아닙니다
            continue
        if not np.allclose(yd, xd, rtol=1e-3, atol=0.02 * width):
            continue                                   # 세로가 가로를 따라가지 않으면 넘어갑니다
        if xy is not None and getattr(xy, "size", 0):
            pt_span = float(xy[:, 0].max() - xy[:, 0].min())
            overlap = float(min(xd.max(), xy[:, 0].max()) - max(xd.min(), xy[:, 0].min()))
            if pt_span > 0 and overlap < 0.4 * pt_span:          # 점이 놓인 구간을 거의 덮지 못하는 토막 선
                continue
        return True
    return False


_ABSMAX_GAP = "선의 절댓값 최댓값이"      # judge 가 이 머리로 「선의 값」 어긋남을 가려 힌트를 따로 냅니다


def _plot_gaps(ax, spec):
    """축 하나를 기준과 대조해 어긋난 곳을 문장 목록으로 돌려줍니다 (빈 목록이면 통과)"""
    gaps = []
    if spec.get("kind") == "line":
        cur, vline = _lines(ax)
        want = spec.get("n_series", len(cur))
        if len(cur) != want:
            gaps.append(f"곡선이 {len(cur)}개다 (기준 {want}개)")
        if cur:
            xd, yd = max(cur, key=lambda t: t[0].size)
            lo, hi = spec.get("x_range", (xd.min(), xd.max()))
            slack = 0.2 * (hi - lo)
            if not (xd.min() <= lo + slack and xd.max() >= hi - slack):
                gaps.append(f"가로축이 {xd.min():.3g}~{xd.max():.3g} 만 포함한다 (기준 {lo:g}~{hi:g})")
            if "peak_x" in spec:
                p0, p1 = spec["peak_x"]
                peak = float(xd[yd.argmax()])
                trough = float(xd[yd.argmin()])
                if not (p0 <= peak <= p1 or p0 <= trough <= p1):
                    gaps.append(f"곡선의 최댓값 위치가 {peak:.4f}, 최솟값 위치가 {trough:.4f} 다"
                                f" (기준 {p0}~{p1} — 둘 중 하나가 이 구간 안에 있어야 합니다)")
        if cur and "absmax" in spec:
            # 선의 값 크기 — 선 개수와 축 이름만 보면 같은 168시각의 다른 값(실측, 예측을 빼지 않은
            # 강건 Z)을 그려도 통과합니다(2026-10-01). 선마다 절댓값 최댓값을 재어 하나라도 기준에 들면 통과입니다
            a0, a1 = spec["absmax"]
            tops = [float(np.nanmax(np.abs(y))) if np.isfinite(y).any() else float("nan")
                    for _, y in cur]
            if not any(a0 <= t <= a1 for t in tops):
                shown = " · ".join(f"{t:.4g}" for t in tops[:3])
                gaps.append(f"{_ABSMAX_GAP} {shown} 다 (기준 {a0:g}~{a1:g})")
        if spec.get("band"):                          # 하한~상한을 채운 띠 — 있는가, 평균 높이가 기준에 닿는가
            polys = [c for c in ax.collections
                     if type(c).__name__ in ("PolyCollection", "FillBetweenPolyCollection")]
            if not polys:
                gaps.append("하한과 상한 사이를 옅은 색으로 채운 구간 띠가 없다 (`fill_between` 으로 채웁니다)")
            elif isinstance(spec["band"], (list, tuple)):
                heights = []
                for c in polys:
                    for p in c.get_paths():
                        v = np.asarray(p.vertices, dtype=float)
                        v = v[np.isfinite(v).all(axis=1)]
                        wide = float(v[:, 0].max() - v[:, 0].min()) if len(v) >= 3 else 0.0
                        if wide > 0:                   # 넓이 ÷ 가로 폭 = 띠의 평균 높이
                            area = 0.5 * abs(float(np.dot(v[:, 0], np.roll(v[:, 1], 1))
                                                   - np.dot(v[:, 1], np.roll(v[:, 0], 1))))
                            heights.append(area / wide)
                b0, b1 = spec["band"]
                if heights and not (b0 <= max(heights) <= b1):
                    gaps.append(f"구간 띠의 평균 높이가 {max(heights):.4g} 다 (기준 {b0:g}~{b1:g})")
        if cur and ("봉우리_x" in spec or "골짜기_x" in spec):
            # 부호가 뜻을 가진 곡선 — 위로 솟는 곡선과 아래로 꺼지는 곡선을 따로 봅니다.
            # 반응을 통째로 뒤집어 그리면 봉우리·골짜기의 자리는 그대로라 자리만 봐서는 걸러지지 않습니다
            if "봉우리_x" in spec:
                a0, a1 = spec["봉우리_x"]
                if not any(float(np.nanmax(y)) > 0 and a0 <= float(x[int(np.nanargmax(y))]) <= a1
                           for x, y in cur):
                    gaps.append(f"위로 솟아 {a0:g}~{a1:g} 시간 구간에서 최댓값에 이르는 곡선이 없다"
                                " (부호가 뒤집혔으면 반응을 반대 방향으로 읽은 것입니다)")
            if "골짜기_x" in spec:
                b0, b1 = spec["골짜기_x"]
                if not any(float(np.nanmin(y)) < 0 and b0 <= float(x[int(np.nanargmin(y))]) <= b1
                           for x, y in cur):
                    gaps.append(f"아래로 꺼져 {b0:g}~{b1:g} 시간 구간에서 최솟값에 이르는 곡선이 없다"
                                " (부호가 뒤집혔으면 반응을 반대 방향으로 읽은 것입니다)")
    elif spec.get("kind") == "bar":
        heights = [float(p.get_height()) for p in ax.patches]
        if "n" in spec and len(heights) != spec["n"]:
            gaps.append(f"막대가 {len(heights)}개다 (기준 {spec['n']}개)")
        if heights and "ymax" in spec and not (spec["ymax"][0] <= max(heights) <= spec["ymax"][1]):
            gaps.append(f"가장 높은 막대가 {max(heights):.4f} 다"
                        f" (기준 {spec['ymax'][0]:g}~{spec['ymax'][1]:g})")
    elif spec.get("kind") == "interval":
        spans = _spans(ax)
        if "n_spans" in spec:
            n_lo, n_hi = spec["n_spans"]
            if not (n_lo <= len(spans) <= n_hi):
                gaps.append(f"가로 구간 막대가 {len(spans)}개다 (기준 {n_lo}개 — 항목 하나에 막대"
                            f" 하나, 강조로 덧그린 막대는 {n_hi - n_lo}개까지 넘어갑니다)")
        if spans:
            widths = [right - left for left, right in spans]
            left_end, right_end = min(left for left, _ in spans), max(right for _, right in spans)
            if "span_max" in spec:
                lo, hi = spec["span_max"]
                if not (lo <= max(widths) <= hi):
                    gaps.append(f"가장 넓은 막대의 폭이 {max(widths):.4f} 다 (기준 {lo:g}~{hi:g})")
            if "span_min" in spec:
                lo, hi = spec["span_min"]
                if not (lo <= min(widths) <= hi):
                    gaps.append(f"가장 좁은 막대의 폭이 {min(widths):.4f} 다 (기준 {lo:g}~{hi:g})")
            if "x_range" in spec:
                lo, hi = spec["x_range"]
                if not (_near(left_end, lo) and _near(right_end, hi)):
                    gaps.append(f"막대가 차지하는 가로축 값이 {left_end:.4g}~{right_end:.4g} 다"
                                f" (기준 {lo:g}~{hi:g})")
        if "n_points" in spec:
            _, pt_sets = _pt_sets(ax)
            n_pts = int(sum(p.shape[0] for p in pt_sets))
            n_lo, n_hi = spec["n_points"]
            if not (n_lo <= n_pts <= n_hi):
                gaps.append(f"점추정 마커가 {n_pts}개다 (기준 {n_lo}개 — 항목 하나에 마커 하나)")
        if "vline_x" in spec:
            _, vline = _lines(ax)
            v0, v1 = spec["vline_x"]
            if not any(v0 <= v <= v1 for v in vline):
                gaps.append(f"기준 구간({v0:g} ~ {v1:g}) 안에 세로선이 없다"
                            + (f" (지금 {vline[0]:.4f})" if vline else " (세로선이 아예 없다)"))
    elif spec.get("kind") == "scatter":
        marker, pt_sets = _pt_sets(ax)
        n_pts = int(sum(p.shape[0] for p in pt_sets))
        if "n_points" in spec:
            cell = spec["n_points"]
            n_lo, n_hi = cell if isinstance(cell, tuple) else (cell, cell)
            if not (n_lo <= n_pts <= n_hi):
                gaps.append(f"점이 {n_pts}개다 (기준 {n_lo}개 — 항목 하나에 점 하나,"
                            f" 강조로 덧찍는 점은 {n_hi - n_lo}개까지 넘어갑니다)")
        xy = np.vstack([p for p in pt_sets if p.size]) if any(p.size for p in pt_sets) else None
        if xy is not None:
            xy = xy[np.isfinite(xy).all(axis=1)]
        if xy is not None and xy.size:
            xs, ys = xy[:, 0], xy[:, 1]
            if "x_range" in spec:
                lo, hi = spec["x_range"]
                if not (_near(xs.min(), lo) and _near(xs.max(), hi)):
                    gaps.append(f"가로축 값이 {xs.min():.3g}~{xs.max():.3g} 다 (기준 {lo:g}~{hi:g})")
            if "y_range" in spec:
                lo, hi = spec["y_range"]
                if not (_near(ys.min(), lo) and _near(ys.max(), hi)):
                    gaps.append(f"세로축 값이 {ys.min():.4g}~{ys.max():.4g} 다 (기준 {lo:g}~{hi:g})")
            if "ymax" in spec and not (spec["ymax"][0] <= ys.max() <= spec["ymax"][1]):
                gaps.append(f"세로축 최댓값이 {ys.max():.4g} 다"
                            f" (기준 {spec['ymax'][0]:g}~{spec['ymax'][1]:g})")
            if "gap_min" in spec:
                spread = float(np.abs(ys - xs).max())
                if spread < spec["gap_min"]:
                    gaps.append(f"두 축의 값이 가장 크게 벌어진 곳도 {spread:.4f} 뿐이다"
                                f" (기준 {spec['gap_min']:g} 이상)")
        if spec.get("xscale") and ax.get_xscale() != spec["xscale"]:
            gaps.append(f"가로축이 {ax.get_xscale()} 눈금이다 (기준 {spec['xscale']} 눈금)")
        if spec.get("diag") and not _diagonal(ax, skip=[ln for ln, _, _ in marker], xy=xy):
            gaps.append("두 축의 값이 같은 점을 잇는 대각선(y = x)이 점이 놓인 구간에 없다")
    if spec.get("축라벨") and not (ax.get_xlabel().strip() and ax.get_ylabel().strip()):
        gaps.append("가로축·세로축 라벨 중 빈 것이 있다")
    return gaps


_REF = {}      # 참조 그림은 두지 않습니다 — 미션마다 완성본 풀이가 그 자리를 대신합니다


def _show(fig):
    """채점 훅 안에서 그림을 화면에 올립니다 (인라인 백엔드는 이 시점에 plt.show() 가 듣지 않습니다)"""
    try:
        from IPython.display import display
        display(fig)
    except Exception:
        plt.show()


def _ref_show(fig, key):
    """내 그림과 참조 그림을 나란히 — 내 그림이 없으면 참조 그림도 띄우지 않습니다"""
    if not _axes(fig) or key not in _REF:
        return
    ref = None
    try:
        ref = _REF[key]()
        if ref is None:
            return
        panels = [(fig, "내 그림"), (ref, "참조 그림")]
        f = _orig_figure(figsize=(FIG_W, 3.4))
        for i, (g, title) in enumerate(panels):
            a = f.add_subplot(1, len(panels), i + 1)
            buf = io.BytesIO()
            (g if hasattr(g, "savefig") else g.figure).savefig(buf, format="png", dpi=110,
                                                               bbox_inches="tight")
            buf.seek(0)
            a.imshow(plt.imread(buf))
            a.axis("off")
            a.set_title(title, fontsize=11)
        _show(f)
        plt.close(f)
    except Exception:
        print("ⓘ 참조 그림을 나란히 놓지 못했습니다 — 위 채점 결과는 그대로 유효합니다")
    finally:
        if ref is not None:
            plt.close(ref)


def _one_line(e, n=120):
    """예외를 한 문장으로 — 트레이스백 대신 학생이 읽을 문장 하나만 남깁니다"""
    txt = str(e).replace("\n", " ").strip()
    return (txt[:n] + "…") if len(txt) > n else txt



def _prompt1(n=70):
    """지금 셀의 지시 첫 행 — 튜터에 보낼 질문에 그대로 싣습니다"""
    head_lines = [l.strip() for l in str(_get("PROMPT") or "").split("\n") if l.strip()]
    return (head_lines[0][:n] + "…") if head_lines else "____"


_PROMPT_MIN = 30
_PROMPT_PLACEHOLDERS = ("", "목표", "쓸 변수", "출력", "목표 —", "쓸 변수 —", "출력 —", "TODO",
                    "여기에 내 말로 적습니다")


def _prompt_missing():
    """지시 입력란이 아직 내 말로 채워지지 않았으면 사유 한 문장, 채워졌으면 None"""
    try:
        txt = str(_ip.user_ns.get("PROMPT", _get("PROMPT")) or "").strip()
    except Exception:
        txt = str(_get("PROMPT") or "").strip()
    if not txt:
        return "PROMPT 입력란이 비어 있습니다"
    if len(txt) < _PROMPT_MIN:
        return f"PROMPT 입력란이 {len(txt)}자뿐입니다 (최소 {_PROMPT_MIN}자)"
    stripped = txt.replace("—", "").replace("·", "").strip()
    if stripped in _PROMPT_PLACEHOLDERS:
        return "PROMPT 입력란이 자리표시자 그대로입니다"
    return None


def _reveal_show(txt):
    """완성본 코드와 해설 — **평문 그대로** 찍는다.

    LMS 렌더러 계약: 「🔓 정답이 열렸습니다 …」 행부터 채점 표를 닫는 `─` 구분선 행까지를
    출력 패널이 「🔓 완성본 풀이 보기」 상자로 접는다. 접힘의 표지가 이 **평문 마커**라
    HTML `details` 로 싣지 않는다 — 마커도 이 함수도 바꾸지 않는다.
    """
    print(txt)


def _flag_side(node):
    """비교식의 한쪽을 지금 이름들로 다시 계산해 수 배열로 — 계산하지 못하거나 수가 아니면 None"""
    try:
        with contextlib.redirect_stdout(io.StringIO()):
            val = eval(compile(ast.Expression(body=node), "<flag>", "eval"), globals())
        arr = np.asarray(val)
        if arr.dtype.kind == "O":
            arr = np.asarray(val, dtype=float)
    except Exception:
        return None
    if arr.dtype.kind not in "iuf":            # 참·거짓 · 날짜 · 글자는 비교한 값이 아닙니다
        return None
    return arr.astype(float)


def _flag_compares(col, size):
    """이 셀 코드에서 `col` 열을 채운 식의 비교마다 (비교한 값 배열들, 비교한 수들) — 못 읽으면 빈 목록.

    `plan["이상"] = z.abs() > ALERT` 처럼 그 열을 채운 대입을 찾습니다(`.loc[조건, "이상"]` 의 조건,
    `.assign(이상=…)` 도 봅니다). 식에 비교가 없고 이름만 있으면(`plan["이상"] = flags`) 그 이름을 채운
    대입으로 세 번까지 거슬러 올라갑니다. 비교(`>`·`.gt()`·`np.greater` 등) 양쪽을 다시 계산해,
    길이가 `size` 인 배열과 수 하나를 가려 돌려줍니다.
    """
    code = _STATE.get("src") or ""
    try:
        code = _ip.transform_cell(code)          # `%`·`!` 행을 파이썬으로 바꿔야 파싱이 걸리지 않습니다
    except Exception:
        pass
    try:
        tree = ast.parse(code)
    except SyntaxError:
        return []

    def _is_col(sl):
        last = sl.elts[-1] if isinstance(sl, ast.Tuple) and sl.elts else sl
        return isinstance(last, ast.Constant) and last.value == col

    stmts = sorted((n for n in ast.walk(tree) if isinstance(n, ast.Assign)),
                   key=lambda n: (n.lineno, n.col_offset))
    roots, named = [], []
    for st in stmts:
        for tgt in st.targets:
            if isinstance(tgt, ast.Subscript) and _is_col(tgt.slice):
                roots += [(st.lineno, st.value), (st.lineno, tgt.slice)]
            elif isinstance(tgt, ast.Name):
                named.append((st.lineno, tgt.id, st.value))
        for n in ast.walk(st.value):
            if isinstance(n, ast.Call):
                roots += [(st.lineno, kw.value) for kw in n.keywords if kw.arg == col]

    def _pairs(expr, line, depth=0):
        found = []
        for n in ast.walk(expr):
            if isinstance(n, ast.Compare):
                found.append([n.left] + list(n.comparators))
            elif isinstance(n, ast.Call) and isinstance(n.func, ast.Attribute):
                if n.func.attr in ("gt", "ge", "lt", "le") and len(n.args) >= 1:
                    found.append([n.func.value, n.args[0]])
                elif n.func.attr in ("greater", "greater_equal", "less", "less_equal") and len(n.args) >= 2:
                    found.append(list(n.args[:2]))
        if found or depth >= 3:
            return found
        for name in sorted({n.id for n in ast.walk(expr) if isinstance(n, ast.Name)}):
            prev = [(ln, v) for ln, nm, v in named if nm == name and ln < line]
            if prev:
                found += _pairs(prev[-1][1], prev[-1][0], depth + 1)
        return found

    out = []
    for line, root in roots:
        for sides in _pairs(root, line):
            arrs, nums = [], []
            for side in sides:
                arr = _flag_side(side)
                if arr is None:
                    continue
                if arr.ndim == 0:
                    nums.append(float(arr))
                elif arr.size == size:
                    arrs.append(arr.reshape(-1))
            if arrs:
                out.append((arrs, nums))
    return out


def _flag_scale(vals, line):
    """비교한 값이 선의 값과 같은 모양인가 — |값| = k × |선| 인 k(선이 |z| 여도, 값을 임계값으로 나눴어도 받습니다), 아니면 None"""
    va, la = np.abs(vals), np.abs(line)
    fin = np.isfinite(va)
    if not np.array_equal(fin, np.isfinite(la)):
        return None
    nz = fin & (la > 1e-12)
    if not nz.any():
        return None
    k = float(np.median(va[nz] / la[nz]))
    if not (np.isfinite(k) and k > 0):
        return None
    return k if np.allclose(va[fin], k * la[fin], rtol=1e-4, atol=1e-9 * max(1.0, k)) else None


def _flag_vs_line(col, marks, lines, thr):
    """`이상` 열과 그래프의 선이 서로 맞는가 — 맞거나 판단할 수 없으면 None, 어긋나면 한 문장.

    고른 임계값 5 에서는 참인 시각이 0개라, 참인 시각 집합만 보면 `이상` 열을 다른 z 로 채워도
    통과합니다(2026-10-01 재현: 전체 중앙값 −35 · MAD 120 으로 계산한 z, 절댓값 최댓값 2.765).
    그래서 두 가지를 봅니다. 선의 |값| 이 임계값을 넘는 시각이 참인 시각과 같은가, 그리고 셀
    코드에서 이 열을 채운 비교가 선과 같은 값을 고른 임계값과 비교했는가입니다.
    """
    k_line = [int((np.abs(y) > thr).sum()) for y in lines]
    if not any(np.array_equal(np.abs(y) > thr, marks) for y in lines):
        return (f"참인 시각 {int(marks.sum())}개가 그래프에 그린 강건 Z 의 절댓값이 {thr:g} 를"
                f" 넘는 시각 {k_line[0]}개와 다릅니다")
    other, off_thr = [], []
    for arrs, nums in _flag_compares(col, marks.size):
        for vals in arrs:
            k = next((s for s in (_flag_scale(vals, y) for y in lines) if s), None)
            if k is None:
                other.append(float(np.nanmax(np.abs(vals))) if np.isfinite(vals).any() else float("nan"))
                continue
            used = [abs(v) / k for v in nums]
            if not used or any(np.isclose(u, thr, rtol=1e-6, atol=1e-9) for u in used):
                return None                      # 선과 같은 값을 고른 임계값과 비교했습니다
            off_thr.append(used[0])
    if off_thr:
        return f"그래프에 그린 강건 Z 를 {off_thr[0]:.4g} 와 비교해 채웠는데 고른 임계값은 {thr:g} 입니다"
    if other:
        top = float(np.nanmax(np.abs(lines[0])))
        return (f"참·거짓을 가른 값(절댓값 최댓값 {other[0]:.4g})이 그래프에 그린 강건 Z"
                f"(절댓값 최댓값 {top:.4g})와 다릅니다")
    return None                                  # 식을 읽지 못한 자리는 집합 대조까지만 봅니다


def _flag_check(spec, line_ys=()):
    """계획표에 적은 참·거짓 열이 고른 임계값으로 탐지한 시각과 같은가 — (통과 여부, 한 문장).

    값과 그래프만 보면 이 열을 다른 식으로 채워도 통과합니다(2026-10-01 검증: 예측을 빼지 않은
    강건 Z 로 `이상` 50개인데 ✅). 그래서 참인 시각을 임계값별 기준 시각과 대조합니다. 고른
    임계값의 이름(`pick`)을 이 셀에서 대입했으면 그 임계값의 시각과 같아야 하고, 이름을 모르면
    기준 임계값 가운데 어느 하나의 시각과 같으면 통과입니다. 그래프가 기준을 통과했으면
    (`line_ys` 에 선의 값이 오면) 그 선과도 맞는지 `_flag_vs_line` 으로 한 번 더 봅니다.
    """
    name, col = spec.get("frame", "plan"), spec.get("col", "이상")
    frame = _get(name)
    if col not in list(getattr(frame, "columns", [])):
        return False, f"`{name}` 에 `{col}` 열이 없습니다"
    try:
        marks = np.array([False if (v is None or (isinstance(v, float) and v != v)) else bool(v)
                          for v in list(frame[col])], dtype=bool)
        got = tuple(sorted(str(t)[:16] for t in frame.index[marks]))
    except Exception as e:
        return False, f"`{col}` 열을 참·거짓으로 읽지 못했습니다 ({type(e).__name__})"
    sets = {float(a): tuple(v) for a, v in spec.get("sets", {}).items()}
    pick, pick_name = None, spec.get("pick", "")
    if pick_name and re.search(rf"(?m)^\s*{re.escape(pick_name)}\s*=(?!=)", _STATE.get("src") or ""):
        try:
            pick = float(_get(pick_name))
        except (TypeError, ValueError):
            pick = None
    if pick in sets:
        if got != sets[pick]:
            return False, (f"참인 시각이 {len(got)}개라 고른 임계값 {pick_name} = {pick:g} 로 탐지한"
                           " 시각과 다릅니다")
        thr, ok_msg = pick, f"고른 임계값 {pick:g} 로 탐지한 시각 {len(got)}개와 같습니다"
    else:
        hit = [a for a, v in sets.items() if got == v]
        if not hit:
            return False, f"참인 시각이 {len(got)}개라 어느 임계값으로 탐지한 시각과도 맞지 않습니다"
        thr, ok_msg = hit[0], f"임계값 {hit[0]:g} 로 탐지한 시각 {len(got)}개와 같습니다"
    lines = []
    for y in line_ys or ():
        try:
            y = np.asarray(y, dtype=float).reshape(-1)
        except (TypeError, ValueError):
            continue
        if y.size == marks.size:
            lines.append(y)
    if lines:
        try:
            why = _flag_vs_line(col, marks, lines, thr)
        except Exception:
            why = None
        if why:
            return False, why
    return True, ok_msg


def _error_hint(key, error):
    """오류 종류에 맞는 (힌트, 물을 것) — 종류별 힌트(`error_kinds`)에 맞는 것이 없으면 미션 공통 힌트입니다.

    공통 힌트는 준비 셀을 건너뛴 NameError 를 먼저 짚습니다. 계산은 돌았는데 고를 값이 없어 난
    오류(빈 목록의 `min()` 등)에 그 힌트를 내면 학생이 애먼 준비 셀을 다시 돌립니다(2026-10-01).
    """
    tbl = _HINT.get(key, {})
    kind, msg = type(error).__name__, str(error)
    for rule in tbl.get("error_kinds", ()):
        if any(kind == k and re.search(pat, msg) for k, pat in rule.get("match", ())):
            return rule.get("hint", ""), rule.get("ask") or "이 오류가 어느 행에서 왜 났는지"
    hint, question = tbl.get("error", ("", "이 오류가 어느 행에서 왜 났는지"))
    return hint, question


def judge(key, text="", figs=(), error=None):
    """출력 텍스트와 그림을 기준과 대조해 채점 표를 찍습니다 — 통과하면 완성본 풀이를 엽니다"""
    spec = _ANS.get(key)
    if spec is None:                       # 기준이 아직 없는 미션은 조용히 넘어갑니다
        return None
    head_label = ("미션 " + key[1:]) if key[:1] == "m" and key[1:].isdigit() else key
    failed, passed = [], True
    print("\n" + "─" * 54)
    print(f"◼ {head_label} 채점 결과")
    reason = _prompt_missing()                     # 값·그래프보다 먼저 — 내 말로 시킨 자리가 있는가
    _locked = bool(reason)
    if reason:
        print(f"❌ {reason} — 짧게 시켜도 됩니다. 다만 무엇을 어떤 단위로 출력할지까지 적으면 한 번에 통과할 확률이 높습니다")
        print("   PROMPT 입력란을 채우기 전에는 값·그래프 채점도, 완성본 풀이도 열지 않습니다")
        print("─" * 54)
        return False
    if error is not None:                   # 오류가 나면 값·그래프 채점은 뜻이 없습니다
        print(f"❌ 실행 오류 — {type(error).__name__}: {_one_line(error)}")
        hint, question = _error_hint(key, error)          # 오류 종류별 힌트가 있으면 그것, 없으면 미션 공통 힌트
        if hint:
            print(f"   힌트 : {hint}")
        form = _ASK.get(key, "full")
        print(_ASK_HEAD[form])
        print("  " + _ASK_FORM[form].format(name="실행 오류", prompt=_prompt1(), ask=question))
        print("─" * 54)
        return False
    txt = text or ""
    if not txt.strip():
        print("❌ 출력이 없습니다 — 값을 print 하도록 프롬프트의 출력 형식에 적으세요")
    units = [str(ref[2]) for ref in spec.get("values", {}).values() if len(ref) > 2]
    val_lines = _value_lines(txt, units)                    # 단위·「=」 가 붙은 행이 있으면 그 행만 봅니다
    nums = _nums("\n".join(val_lines)) if val_lines else _nums(txt)
    table_flood = not val_lines and len(nums) > 30        # 골라낸 자리 없이 숫자만 잔뜩 = 표를 통째로 띄운 모양
    if table_flood:
        print(f"❌ 값 — 단위도 「이름 = 값」 도 없이 숫자가 {len(nums)}개 나왔습니다 — 표를"
              " 통째로 출력하면 어느 숫자가 답인지 채점 코드가 판별하지 못합니다."
              " 값은 「이름 = 숫자 단위」 한 행으로 출력하세요")
        nums = []
    for label, ref in spec.get("values", {}).items():
        lo, hi = float(ref[0]), float(ref[1])
        unit = ref[2] if len(ref) > 2 else ""
        tail = _label_lines(val_lines, label) if nums else []   # 이 라벨이 든 행의 「=」 뒤만 — 옆 칸과 안 섞이게
        fell_back = bool(nums) and not tail             # 라벨이 든 행이 없을 때만 값줄 전체로 물러납니다
        picked = _tail_numbers(tail) if tail else nums   # 꼬리는 괄호를 걷어낸 첫 숫자 하나만 봅니다
        hits = [v for v in picked if lo <= v <= hi]
        if hits:
            print(f"✅ {label} — 출력에서 {hits[0]:,.6g} 를 찾았습니다")
            if unit and unit not in txt:
                print(f"   ⓘ 단위 「{unit}」 도 값과 함께 출력하면 보고서에 그대로 옮길 수 있습니다")
        elif table_flood:
            passed = False
            failed.append((label, label))
            print(f"❌ {label} — 「이름 = 숫자 {unit}」 한 행으로 골라 출력하면 채점 코드가 읽습니다")
        else:
            passed = False
            failed.append((label, label))
            _prev_fail = _TRY.get(key, 0)              # 결정 16 — 첫 ❌ 에는 기준 범위를 감춘다
            print(f"❌ {label} — 기준에 닿는 숫자를 출력에서 찾지 못했습니다"
                  + (f" (기준 범위 {lo:g} ~ {hi:g})" if _prev_fail else ""))
            if fell_back:                            # 라벨 행을 못 찾아 값줄 전체를 본 자리
                print(f"   ⓘ 라벨 「{label}」 이 든 행을 찾지 못해 값줄 전체에서 찾았습니다"
                      " — 「이름 = 숫자 단위」 한 행으로 출력하면 그 행만 보고 채점합니다")
    plot_spec = spec.get("plot")
    good_ax = None                          # 기준을 통과한 축 — `이상` 열을 그 선과 대조할 때 씁니다
    if plot_spec:
        axs = [a for f in figs for a in _axes(f)]
        if not axs:
            passed = False
            failed.append(("그래프", "plot"))
            print("❌ 그래프 — 그림을 찾지 못했습니다 (그래프 한 장도 같은 프롬프트에 함께 시킵니다)")
        else:
            gaps = None
            for ax in axs:                 # 패널이 여럿이면 기준에 맞는 축이 하나만 있어도 통과입니다
                try:
                    this_gaps = _plot_gaps(ax, plot_spec)
                except Exception as e:
                    this_gaps = [f"그림에서 데이터를 뽑다 걸렸습니다 ({type(e).__name__})"]
                if not this_gaps:
                    gaps = []
                    good_ax = ax
                    break
                if gaps is None or len(this_gaps) < len(gaps):
                    gaps = this_gaps
            if gaps:
                passed = False
                size_gaps = [g for g in gaps if g.startswith(_ABSMAX_GAP)]
                if len(size_gaps) < len(gaps):              # 선 개수 · 축 이름 같은 모양 문제
                    failed.append(("그래프", "plot"))
                if size_gaps:                               # 선에 그린 값이 기준 크기가 아닌 문제 — 힌트를 따로 냅니다
                    failed.append((plot_spec.get("absmax_name", "그래프의 선 값"),
                                   "plot_z" if "plot_z" in _HINT.get(key, {}) else "plot"))
                print("❌ 그래프 — " + " · ".join(gaps))
            else:
                print("✅ 그래프 — 기준대로 그렸습니다")
            _ref_show(figs[0] if figs else None, key)
    flag_spec = spec.get("flag_col")
    if flag_spec:                          # 계획표의 참·거짓 열 — 값과 그래프가 맞아도 이 열은 따로 틀릴 수 있습니다
        flag_name = f"`{flag_spec.get('col', '')}` 열"
        flag_lines = []
        if good_ax is not None:            # 그래프가 통과했을 때만 선과 대조합니다 — 선이 틀렸으면 그래프 힌트가 먼저입니다
            try:
                flag_lines = [y for _, y in _lines(good_ax)[0]]
            except Exception:
                flag_lines = []
        flag_ok, why = _flag_check(flag_spec, flag_lines)
        if flag_ok:
            print(f"✅ {flag_name} — {why}")
        else:
            passed = False
            failed.append((flag_name, "flag"))
            print(f"❌ {flag_name} — {why}")
    tbl = _HINT.get(key, {})
    if failed:
        _TRY[key] = _TRY.get(key, 0) + 1     # 결정 16 — 다음 ❌ 부터 기준 범위를 인쇄합니다
    for name, hkey in failed:
        hint = tbl.get(hkey, ("",))[0]
        first_tier = (hint,) if isinstance(hint, str) else tuple(hint)   # 함정 미션은 1차·2차로 나뉩니다
        for tiers, txt in enumerate(first_tier, 1):
            if txt:
                tail = f" {tiers}단" if len(first_tier) > 1 else ""
                print(f"   힌트({name}){tail} : {txt}")
    if failed:
        form = _ASK.get(key, "full")
        print(_ASK_HEAD[form])
        for name, hkey in failed:
            question = (tbl.get(hkey, ("", ""))[1] if len(tbl.get(hkey, ("",))) > 1 else "") \
                or "이 항목이 왜 ❌ 로 남는지"
            print("  " + _ASK_FORM[form].format(name=name, prompt=_prompt1(), ask=question))
    elif _locked:                                  # 값·그림은 닿았지만 내 말로 시킨 자리가 아직 비었습니다
        print("ⓘ 값과 그림은 기준에 닿았습니다 — PROMPT 에 고칠 지시를 내 말로 쓰고"
              " 다시 실행하면 완성본 풀이가 열립니다")
    else:
        print("🔓 정답이 열렸습니다 — 아래 「🔓 완성본 풀이 보기」를 펼쳐 내 코드와 견줘 보세요")
        _reveal_show(base64.b64decode(_REVEAL[key]).decode("utf-8"))
    print("─" * 54)
    return passed and not _locked


class _Tee:
    """「PROMPT 셀」의 출력을 화면과 버퍼 양쪽으로 흘려보냅니다"""

    def __init__(self, orig, buf):
        self._o, self._b = orig, buf

    def write(self, s):
        self._b.write(s)
        return self._o.write(s)

    def flush(self):
        self._o.flush()

    def __getattr__(self, n):
        return getattr(self._o, n)


def _tag(src):
    """첫 행이 `# 미션 N — 프롬프트`·`# 과제 — 프롬프트` 인 셀만 채점 대상입니다"""
    head_line = (src or "").split("\n")[0]
    m = re.match(r"\s*#\s*(미션\s*(\d+)|과제)\b", head_line)
    if not m or "프롬프트" not in head_line:                           # 「준비 셀」(`# 미션 N · 준비`)은 대상이 아닙니다
        return None
    return "m" + m.group(2) if m.group(2) else "과제"


def _pre(info=None):
    """「PROMPT 셀」이면 출력 버퍼를 걸고 그림 목록을 비웁니다"""
    try:
        tag = _tag(getattr(info, "raw_cell", ""))
        _STATE["tag"] = tag
        _STATE["src"] = str(getattr(info, "raw_cell", "") or "")   # 고른 임계값을 이 셀에서 대입했는지 볼 때 씁니다
        if tag is None or tag not in _ANS:
            return
        _FIGS.clear()
        _STATE["ids0"] = {id(f) for f in _open_figs()}
        _STATE["buf"], _STATE["out"] = io.StringIO(), sys.stdout
        sys.stdout = _Tee(sys.stdout, _STATE["buf"])
    except Exception:
        pass


def _post(result=None):
    """「PROMPT 셀」이 끝나면 출력을 되돌리고 그 셀의 출력·그림으로 채점합니다"""
    tag, buf, out = _STATE.get("tag"), _STATE.get("buf"), _STATE.get("out")
    _STATE["tag"], _STATE["buf"], _STATE["out"] = None, None, None
    if out is not None:
        sys.stdout = out
    if tag is None or tag not in _ANS:
        return
    seen = {id(f) for f in _FIGS}
    figs_all = list(_FIGS) + [f for f in _open_figs()
                         if id(f) not in seen and id(f) not in _STATE.get("ids0", set())]
    txt = buf.getvalue() if buf is not None else ""
    last_val = getattr(result, "result", None)       # 셀 마지막 표현식의 값 — print 없이 표로 띄운 값도 읽습니다
    if last_val is not None:
        try:
            txt = txt + "\n" + str(last_val)
        except Exception:
            pass
    try:
        judge(tag, txt, figs_all, error=getattr(result, "error_in_exec", None))
    except Exception as e:
        print(f"(채점을 마치지 못했습니다 — {type(e).__name__}: {_one_line(e)})")


try:
    _ip = get_ipython()
    for _ev, _fn in (("pre_run_cell", _pre), ("post_run_cell", _post)):
        for _old in list(_ip.events.callbacks[_ev]):
            if getattr(_old, "__name__", "") == _fn.__name__:
                _ip.events.unregister(_ev, _old)     # 이 셀을 다시 실행해도 채점은 한 번만 찍힙니다
        _ip.events.register(_ev, _fn)
    print("채점 준비 완료 — 「PROMPT 셀」을 실행하면 실행 결과(출력)에 채점 결과가 찍힙니다")
except Exception:
    print("(IPython 밖입니다 — 훅 없이 judge(태그, 출력, 그림들) 만 준비했습니다)")

## 서울 공유자전거 준비

자료는 **서울 공유자전거 시간당 대여 기록**(UCI 머신러닝 저장소)입니다. 2017년 12월 1일부터 2018년 11월 30일까지 8,760행입니다. 휴무 295시간을 1주일 전 같은 시각 값으로 채운 보정 수요 `demand` 를 씁니다. 학습 구간은 9월 1일～11월 16일 1,848시간이고, 검증 구간은 11월 17～23일 168시간입니다.

「오늘의 준비」 셀은 셋째 날 SARIMA 를 한 번 적합해 10～20초 걸리고, 아래를 출력합니다.

- 표 1 · 단순 예측 셋의 예측 원점 5개 RMSE (대)
- 그림 1 · SARIMA 잔차의 자기상관
- 표 2 · SARIMA 잔차 진단 (잔차 1,824개)
- 표 3 · SARIMA 예측구간이 담은 검증 168시간 (시간)
- 표 4 · 학습 구간과 검증 구간의 기온 (°C)
- 표 5 · 기준 구간 시각별 잔차 중앙값과 MAD (대)

표 1 의 계절 나이브 24 는 원점 전 마지막 하루 24시각을 7일 동안 되풀이하는 예측입니다. 시각별 평균은 9월 1일부터 원점까지 같은 시각 값의 평균을 7일 동안 되풀이합니다. 표 4 의 5백분위수는 크기순으로 정렬한 값에서 아래로부터 5% 지점에 있는 값입니다. 표 5 의 잔차는 실측에서 1주일 전 같은 시각의 보정 수요를 뺀 값입니다.

In [ ]:
# 오늘의 준비 — 읽고 실행만 합니다
# 서울 공유자전거로 표 1～5 와 그림 1 을 만듭니다. SARIMA 를 한 번 적합해 10～20초 걸립니다
URL = "https://archive.ics.uci.edu/static/public/560/seoul+bike+sharing+demand.zip"
#  받아지지 않으면: UCI 페이지에서 zip 을 내려받아 Colab 좌측 파일창에 올린 뒤
#  URL 값으로 파일 이름 "seoul+bike+sharing+demand.zip" 을 적으세요
raw = pd.read_csv(URL, encoding="cp1252", compression="zip")        # 8,760행 x 14열
raw.columns = ["date", "count", "hour", "temp", "humidity", "wind", "visibility",      # 위치 기반 리네임
               "dewpoint", "solar", "rain", "snow", "season", "holiday", "functioning"]
raw["datetime"] = (pd.to_datetime(raw["date"], format="%d/%m/%Y")    # 01/12/2017 = 2017년 12월 1일
                   + pd.to_timedelta(raw["hour"], unit="h"))
df = raw.sort_values("datetime").set_index("datetime")
df.index.freq = "h"                                                 # 등간격 1시간
_s = df["count"].astype(float).mask(df["functioning"].eq("No"))    # 휴무는 0 이 아니라 관측 없음
_y = _s.fillna(_s.shift(168))                                       # 1주일 전 같은 시각 값으로 채운다
df["demand"] = _y.fillna(_y.shift(-168))                            # 1주일 전도 휴무면 1주일 뒤 값으로
train = df.loc["2018-09-01 00:00":"2018-11-16 23:00"]              # 학습 1,848시간
valid = df.loc["2018-11-17 00:00":"2018-11-23 23:00"]              # 검증 168시간

# 표 1: 단순 예측 셋을 예측 원점 5개(금요일 23시)에서 168시간씩 채점한다
ORIGINS = pd.date_range("2018-10-19 23:00", "2018-11-16 23:00", freq="7D")
dem_all = df["demand"].astype(float)
origin_rows = {}
for o in ORIGINS:
    week = pd.date_range(o + pd.Timedelta(hours=1), periods=168, freq="h")   # 원점 1시간 뒤부터 168시간
    last_day = dem_all.loc[o - pd.Timedelta(hours=23):o].to_numpy()          # 원점 전 마지막 하루 24시각
    fit_part = dem_all.loc["2018-09-01 00:00":o]
    hour_mean = fit_part.groupby(fit_part.index.hour).mean()                  # 9월 1일～원점의 시각별 평균
    origin_rows[f"{o:%m-%d} 23시"] = {
        "계절 나이브 168": rmse(dem_all.loc[week], dem_all.shift(168).loc[week]),
        "계절 나이브 24": rmse(dem_all.loc[week], np.tile(last_day, 7)),
        "시각별 평균": rmse(dem_all.loc[week], hour_mean.loc[week.hour].to_numpy())}
origin_rmse = pd.DataFrame(origin_rows).T
origin_rmse.loc["원점 5개 평균"] = origin_rmse.mean()
print("표 1 · 단순 예측 셋의 예측 원점 5개 RMSE (대)")
print(origin_rmse.round(2).to_string())

# 그림 1 · 표 2: 학습 1,848시간에 셋째 날 SARIMA 를 적합하고 잔차를 진단한다
sarima_res = SARIMAX(train["demand"], order=(1, 0, 1), seasonal_order=(1, 1, 1, 24)).fit(disp=False)
sarima_resid = sarima_res.resid.iloc[24:]                          # 앞 24시각은 계절차분 초기값이라 뺀다
resid_acf = acf(sarima_resid, nlags=336)
acf_bound = 1.96 / np.sqrt(len(sarima_resid))
resid_top3 = (np.argsort(-np.abs(resid_acf[1:]))[:3] + 1).tolist()  # 절댓값이 큰 시차 셋
lb_p = acorr_ljungbox(sarima_resid, lags=[24, 168], model_df=4)["lb_pvalue"].to_numpy()
fig, ax = plt.subplots(figsize=(FIG_W, 3.2))
ax.bar(range(1, 337), resid_acf[1:], width=0.8, color=COL["파랑"])
ax.axhline(acf_bound, color=COL["회색"], ls="--", lw=1)
ax.axhline(-acf_bound, color=COL["회색"], ls="--", lw=1)
ax.set_xticks([1, 24, 48, 72, 168, 336])
ax.set_title("그림 1 · SARIMA 잔차의 자기상관")
ax.set_xlabel("시차 (시간)")
ax.set_ylabel("자기상관")
plt.show()
print("표 2 · SARIMA 잔차 진단 (잔차 1,824개)")
print(f"  경계선 ±{acf_bound:.3f} · 시차 1～336 가운데 경계선 밖 {int((np.abs(resid_acf[1:]) > acf_bound).sum())}개")
print("  자기상관이 큰 시차 셋: " + " · ".join(f"{k} ({resid_acf[k]:.3f})" for k in resid_top3))
print(f"  시차 1 · 24 의 자기상관: {resid_acf[1]:.3f} · {resid_acf[24]:.3f}")
print(f"  Ljung-Box p 값: 시차 24 {lb_p[0]:.1e} · 시차 168 {lb_p[1]:.1e}")

# 표 3: 같은 SARIMA 로 검증 168시간을 한 번에 예측하고 예측구간이 실측을 몇 시각 담았는지 센다
sarima_fc = sarima_res.get_forecast(steps=len(valid))
valid_act = valid["demand"].to_numpy()
cover_rows = {}
for nominal, alpha in [("명목 95%", 0.05), ("명목 80%", 0.20)]:       # alpha 는 유의수준, 명목 포함률 = 1 − alpha
    ci = sarima_fc.conf_int(alpha=alpha)
    lo, hi = ci.iloc[:, 0].to_numpy(), ci.iloc[:, 1].to_numpy()
    n_in = int(((valid_act >= lo) & (valid_act <= hi)).sum())
    cover_rows[nominal] = {"담은 시각": n_in, "상한 위": int((valid_act > hi).sum()),
                           "하한 아래": int((valid_act < lo).sum()),
                           "실제 포함률(%)": round(100 * n_in / len(valid_act), 1)}
cover_table = pd.DataFrame.from_dict(cover_rows, orient="index")   # 행 = 명목 포함률
pred_mean = float(sarima_fc.predicted_mean.mean())
print("표 3 · SARIMA 예측구간이 담은 검증 168시간 (시간)")
print(cover_table.to_string())
print(f"  검증 168시간 평균: 실측 {valid_act.mean():.1f}대 · 점 예측 {pred_mean:.1f}대")

# 표 4: 검증 168시간의 기온이 학습 1,848시간 기온 범위 안인지 본다
temp_train = train["temp"].astype(float)
temp_valid = valid["temp"].astype(float)
temp_p5 = float(np.percentile(temp_train, 5))                      # 학습 기온의 5백분위수
n_cold = int((temp_valid < temp_p5).sum())
n_below_min = int((temp_valid < temp_train.min()).sum())
print("표 4 · 학습 구간과 검증 구간의 기온 (°C)")
print(f"  학습 1,848시간: 평균 {temp_train.mean():.1f} · 5백분위수 {temp_p5:.1f} · 최저 {temp_train.min():.1f}")
print(f"  검증 168시간: 평균 {temp_valid.mean():.1f} · 최저 {temp_valid.min():.1f}")
print(f"  검증 168시간 가운데 학습 5백분위수보다 추운 시각 {n_cold}시간")
print(f"  검증 168시간 가운데 학습 최저보다 추운 시각 {n_below_min}시간")

# 표 5: 기준 구간(10월 20일～11월 16일)의 라벨 없는 잔차로 시각별 중앙값과 MAD 를 구한다
resid_base = df["count"].astype(float) - dem_all.shift(168)        # 잔차 = 실측 − 1주일 전 보정 수요
label_hour = df["functioning"].eq("No") | (df["rain"].astype(float) >= 10)   # 정답 라벨: 휴무 · 시간당 강수 10mm 이상
in_base = pd.Series((df.index >= pd.Timestamp("2018-10-20 00:00"))
                    & (df.index <= pd.Timestamp("2018-11-16 23:00")), index=df.index)
base_part = pd.DataFrame({"r": resid_base, "h": df.index.hour}, index=df.index)[
    in_base & ~label_hour & resid_base.notna()]
med_by_hour = base_part.groupby("h")["r"].median()                  # 시각별 잔차 중앙값 24개
mad_by_hour = base_part.groupby("h")["r"].apply(lambda s: float(np.median(np.abs(s - s.median()))))
hour_table = pd.DataFrame({"잔차 중앙값": med_by_hour, "MAD": mad_by_hour}).loc[[0, 4, 8, 13, 18]]
hour_table.index = [f"{h}시" for h in hour_table.index]
print("표 5 · 기준 구간 시각별 잔차 중앙값과 MAD (대)")
print(f"  기준 구간 10월 20일～11월 16일 · 라벨 없는 잔차 {len(base_part)}시각")
print(hour_table.round(0).astype(int).to_string())

## 시계열 모듈 결정 규칙 확인

아래 다섯 문항은 1～5일차에서 한 문항씩 나옵니다. 문항마다 레퍼런스 카드의 결정 규칙을 새 상황에 적용합니다. 코드를 실행하지 않고 풀고, 막히면 그날 카드의 「결론」과 「흔한 실수」를 봅니다.

> **[문제] 평균이 한 방향으로 옮겨 가는 일 매출에는 어떤 차분을 할까요?**
>
> 예를 들어 어느 매장 일 매출 365일이 1년 내내 꾸준히 는다고 합시다. ADF 는 단위근 가설을 기각하지 못했고, KPSS 는 정상 가설을 기각했습니다. 자기상관은 시차 7·14·21 에서 다시 커지지 않고 모든 시차에서 느리게 감소합니다.
> 
> **핵심 키워드:** 두 검정의 기각 방향, 느리게 감소하는 자기상관, 1차 차분 1회
>
> 1. 1차 차분을 1회 한다
> 2. 주기 7 로 계절차분을 1회 한다
> 3. 두 검정의 결론이 엇갈리니 차분하지 않는다

> **[문제] 세 후보의 AIC 로 공식 차수를 고른다면 무엇일까요?**
>
> 예를 들어 첫날 결론이 d = 0 인 계열에서 후보 셋의 AIC 를 구했다고 합시다. (1,0,1) 2,000.4 · (2,0,2) 1,999.1 · (1,1,1) 1,950.2 입니다.
> 
> **핵심 키워드:** 같은 d 끼리만 비교, 차이 2 이하는 항이 적은 쪽, 1위와 2위의 차이
>
> 1. (1,1,1): 세 후보 가운데 AIC 가 가장 낮다
> 2. (2,0,2): d = 0 인 후보 가운데 AIC 가 가장 낮다
> 3. (1,0,1): d = 0 인 두 후보의 차이가 2 이하라 항이 적다

> **[문제] 금요일 23시에 168시간을 예측하는 베이스라인은 어느 코드로 만들까요?**
>
> 예를 들어 금요일 23시까지의 실측만 보고 다음 168시간의 시간당 대여 수를 예측하는 베이스라인을 만든다고 합시다.
> 
> **핵심 키워드:** 원점 전 값만 쓰는 베이스라인, 원점 뒤 실측, 계절 나이브 168
>
> 1. `df["demand"].shift(24)`: 하루 전 같은 시각 값
> 2. `df["demand"].shift(168)`: 1주일 전 같은 시각 값
> 3. `df["demand"].shift(1)`: 한 시간 전 값

> **[문제] 백분율 오차를 한 수로 적을 때 무엇을 쓸까요?**
>
> 예를 들어 3시 실측 10대를 20대로, 18시 실측 1,000대를 1,300대로 예측했다고 합시다.
> 
> **핵심 키워드:** 실측으로 나누는 MAPE, 합으로 나누는 WAPE, 대표 지표 RMSE
>
> 1. MAPE 65%: 두 시각의 오차 비율 100% 와 30% 의 평균
> 2. 3시를 뺀 MAPE 30%: 실측이 작은 시각은 계산에서 뺀다
> 3. WAPE 약 31%: 오차 절댓값의 합 310대 ÷ 실측의 합 1,010대

> **[문제] 원점 5개에서 베이스라인보다 낮게 나온 Prophet 구성은 어떻게 할까요?**
>
> 예를 들어 조건부 계절성을 둔 Prophet 의 원점 5개 평균 RMSE 가 계절 나이브 168 보다 약 3% 낮다고 합시다. 11월 24～30일 테스트 구간은 아직 쓰지 않았습니다.
> 
> **핵심 키워드:** 테스트 구간은 한 번, 여러 후보의 최솟값, 정하기 전 운영 예측은 베이스라인
>
> 1. 곧바로 다음 주 운영 예측을 이 구성으로 바꾼다
> 2. 떼어 둔 테스트 구간에서 한 번 채점한 뒤 정한다
> 3. 테스트 구간에서 구성을 바꿔 가며 가장 낮은 값을 고른다

## 서울 공유자전거로 판단하기

아래 다섯 문항은 「오늘의 준비」 셀 출력을 읽고 답합니다. 표 1 은 단순 예측 셋의 평가표입니다. 그림 1 과 표 2·3 으로는 셋째 날 SARIMA 를 믿어도 되는지 봅니다. 표 4 는 기온, 표 5 는 이상 탐지 규칙에 쓰는 시각별 값입니다.

<img src="https://otexts.com/fpp3/fpp_files/figure-html/tsresiduals-1.png" alt="잔차 시간 그래프, 잔차 자기상관 막대, 잔차 히스토그램을 한 장에 모은 그림" width="560" height="346"/>
<p align="center"><em>▲ 구글 주가에 나이브 기법을 쓴 뒤의 잔차 진단 그림. 위는 잔차를 시간 순으로, 아래 왼쪽은 잔차 자기상관, 오른쪽은 잔차 히스토그램이다. 그림 1 은 아래 왼쪽과 같은 자기상관 막대이고, 가로축이 시간이라 시차 168 이 1주일이다 (출처: Hyndman &amp; Athanasopoulos, Forecasting: Principles and Practice 3판, 그림 5.13)</em></p>

> **[문제] 단순 예측 셋 가운데 운영 예측으로 낼 것**
>
> 「오늘의 준비」 셀을 실행하면 나오는 「표 1 · 단순 예측 셋의 예측 원점 5개 RMSE (대)」를 봅니다. 다음 주 운영 예측으로 어느 것을 낼까요?
> 
> **핵심 키워드:** 원점 5개 평균 RMSE, 이긴 횟수가 아닌 평균, 같은 원점으로 한 열씩
>
> 1. 계절 나이브 168: 원점 5개 평균 RMSE 가 셋 가운데 가장 작다
> 2. 계절 나이브 24: 첫날 계절성 강도는 주기 24 가 주기 7 보다 컸다
> 3. 시각별 평균: 원점 5개 가운데 두 번은 계절 나이브 168 보다 덜 빗나갔다

> **[문제] SARIMA 잔차에 남은 규칙**
>
> 「오늘의 준비」 셀을 실행하면 나오는 「그림 1 · SARIMA 잔차의 자기상관」과 「표 2 · SARIMA 잔차 진단」을 봅니다. 두 출력으로 어떤 결론을 적을까요?
> 
> **핵심 키워드:** 시차 168 잔차 자기상관, p 값이 작으면 규칙이 남았다, 계절 주기 하나인 SARIMA
>
> 1. 하루 주기가 남았다. 계절 주기 24 로 계절차분을 한 번 더 한다
> 2. 1주일 전 같은 시각과의 관계가 남았다. SARIMA 의 한계로 적는다
> 3. Ljung-Box p 값이 0 에 가까우니 잔차는 백색잡음이다. 진단을 통과했다

> **[문제] SARIMA 80% 예측구간을 적는 법**
>
> 「오늘의 준비」 셀을 실행하면 나오는 「표 3 · SARIMA 예측구간이 담은 검증 168시간 (시간)」을 봅니다. 80% 예측구간을 보고서에 어떻게 적을까요?
> 
> **핵심 키워드:** 벗어난 방향, 예측선의 치우침, 명목 포함률과 실제 포함률
>
> 1. 실제 포함률이 명목 80% 와 비슷하니 구간을 그대로 믿고 적는다
> 2. 명목 포함률을 95% 로 높여 벗어나는 시각을 줄인 뒤 적는다
> 3. 벗어난 시각이 모두 하한 아래라 예측선이 높다. 구간은 참고로만 적는다

> **[문제] 기온을 외생 변수로 더하기 전에**
>
> 「오늘의 준비」 셀을 실행하면 나오는 「표 4 · 학습 구간과 검증 구간의 기온 (°C)」를 봅니다. 다음 주 예측에 기온을 외생 변수로 더해도 될까요?
> 
> **핵심 키워드:** 학습 범위 밖의 외생 변수, 5백분위수, 검증 오차로 정하기
>
> 1. 바로 더하지 않는다. 학습에서 드문 기온이 많아 검증 오차로 먼저 확인한다
> 2. 더한다. 기상청 예보로 다음 168시간의 기온을 채울 수 있다
> 3. 더한다. 추운 시각이 많을수록 기온 항이 수요 감소를 잘 담는다

> **[문제] 같은 잔차가 이상으로 탐지되는 시각**
>
> 「오늘의 준비」 셀을 실행하면 나오는 「표 5 · 기준 구간 시각별 잔차 중앙값과 MAD (대)」를 봅니다. 예를 들어 4시·8시·18시의 잔차가 모두 −150대라면 임계값 3.5 에서 이상으로 탐지되는 시각은?
> 
> **핵심 키워드:** 시각별 중앙값과 MAD, 새벽의 작은 MAD, 이상으로 탐지된 시각의 실측과 예측
>
> 1. 18시만
> 2. 4시만
> 3. 세 시각 모두
> 4. 어느 시각도 이상으로 탐지되지 않는다

## 미션 1 — 예측 원점 5개 평가표와 채택 예측

표 1 의 계절 나이브 168 과 후보 두 모형을 같은 예측 원점 5개에서 RMSE 로 다시 채점해, 다음 주 운영 예측을 정합니다. 후보 두 모형은 셋째 날 SARIMA 와 다섯째 날 Prophet 기본 구성입니다. 둘째 날 ARIMA 는 원점 5개 평균 RMSE 가 셋째 날 SARIMA 보다 커서 후보에서 뺍니다. 원점마다 가장 작게 빗나간 예측을 적습니다.

#### 할 일 — 미션 1 (40분)

1. 아래 「미션 1 준비 셀」을 실행합니다. 두 후보를 원점마다 **재적합(refit)** 해서 1분 안팎 걸립니다. 재적합은 9월 1일부터 원점까지로 계수를 다시 추정하는 일입니다. 이 셀은 고치지 않습니다.
2. 아래 「만들 것」대로 「PROMPT 셀」의 PROMPT 에 시킬 일을 내 말로 적습니다.
3. 셀을 클릭한 채 AI 튜터에게 "위 PROMPT 대로 코드를 써 줘" → 「이 셀에 넣기」 → ▶ 합니다. LMS 밖이면 받은 코드를 AI 코드 표시선 아래에 둡니다.
4. 채점 결과를 봅니다. ❌ 면 힌트대로 PROMPT 를 고쳐 3번부터 다시 합니다.

**이 미션의 질문**: 예측 원점 5개로 채점하면 후보 두 모형 가운데 계절 나이브 168 보다 덜 빗나가는 모형이 있을까요?

**쓸 수 있는 것**: 「미션 1 준비 셀」이 만들어 둔 이름입니다.

| 이름 | 무엇 |
|---|---|
| `df` | 서울 공유자전거 시간 표. 열 `demand` 가 보정 수요(대) |
| `ORIGINS` · `CAND_NAMES` | 예측 원점 5개(10월 19일～11월 16일의 금요일 23시)와 후보 이름 2개 |
| `FC` | 후보별 예측을 담은 사전. `FC[이름][k]` 는 k번째 원점 1시간 뒤부터 168시간의 예측이고, 인덱스가 그 168시각이다 |
| `rmse(a, b)` | 두 계열의 RMSE 를 구하는 함수(설치 셀) |

#### 만들 것

굵은 이름은 채점이 찾으니 글자 그대로 씁니다.

1. **채택 예측의 원점 5개 평균 RMSE**: 단위 대 · 소수 둘째 자리. 원점마다 그 168시간의 실측 `demand` 와 예측 셋의 RMSE 를 구합니다. 예측 셋은 `df["demand"].shift(168)` 로 꺼낸 계절 나이브 168 과 후보 2개입니다. 원점 5개를 평균해 가장 작은 예측의 값을 적습니다. 원점별 세 값과 가장 작은 예측 이름도 한 행씩 출력합니다
2. **베이스라인보다 원점 5개 평균 RMSE 가 작은 후보 수**: 단위 개 · 정수
3. **보고 한 행**: 채택한 예측 이름과 채점 조건(예측 원점 5개 · 학습 9월 1일～원점 · 예측 기간 168시간 · 지표 RMSE). 채점하지 않습니다
4. **그래프 한 장**: 원점 5개 평균 RMSE 막대 3개. 가로축 「예측」, 세로축 「원점 5개 평균 RMSE (대)」

**통과 기준**: 1·2번이 「이름 = 값 단위」 한 행씩 출력되고 4번 막대그래프가 있으면 통과입니다.

<img src="https://skforecast.org/latest/img/backtesting_refit.gif" alt="학습 구간이 늘고 채점 구간이 뒤따르는 애니메이션" width="640" height="284"/>
<p align="center"><em>▲ 예측 원점을 옮기며 재적합하고 바로 뒤 구간을 채점하는 방법 (출처: skforecast 공식 문서)</em></p>

#### 프롬프트에 넣을 것(힌트)

- **고정할 이름**: 위 표의 이름은 다시 만들지 말고, SARIMA 와 Prophet 도 재적합하지 말라고 적습니다.
- **채점 시각**: 원점마다 실측과 세 예측을 `FC[이름][k].index` 의 같은 168시각에서 비교하라고 적습니다.
- **베이스라인**: 계절 나이브 168 도 같은 168시각에서 구해 함께 평균하라고 적습니다.
- **출력 형식**: 값 두 개는 「이름 = 값 단위」 한 행씩 출력하라고 적습니다.

<details class="lms-extra"><summary>자세히 — 예시 프롬프트</summary>

위 표의 이름은 새로 만들지 말고, SARIMA 와 Prophet 도 재적합하지 마. 원점마다 FC 예측의 인덱스인 168시각에서 실측 df["demand"] 와 df["demand"].shift(168), 두 후보의 RMSE 를 rmse 로 구해 세 값과 가장 작은 예측 이름을 한 행씩 출력해 줘. 원점 5개 평균이 가장 작은 예측을 채택해 "채택 예측의 원점 5개 평균 RMSE = 값 대" 와 "베이스라인보다 원점 5개 평균 RMSE 가 작은 후보 수 = 값 개" 를 한 행씩, 채택 예측 이름과 채점 조건을 한 행 출력해 줘. 세 예측의 평균 RMSE 를 막대 3개로 그리고 가로축 「예측」, 세로축 「원점 5개 평균 RMSE (대)」 이름을 달아 줘.

</details>

#### 예상 적기

> **[예측] 원점 5개 평균 RMSE 가 가장 작은 예측은 무엇일까요?**
>
> 1. 셋째 날 SARIMA
> 2. 다섯째 날 Prophet 기본 구성
> 3. 계절 나이브 168
>
> 답을 정한 뒤 아래를 읽어 보세요(정답 채점은 없습니다).

In [2]:
# 미션 1 · 준비 — 읽고 실행만 합니다. 이 셀이 미션 1 에 쓸 이름을 직접 만듭니다
# [heavy] 셋째 날 SARIMA 를 원점마다 재적합해 1분 안팎 걸립니다
URL = "https://archive.ics.uci.edu/static/public/560/seoul+bike+sharing+demand.zip"
raw = pd.read_csv(URL, encoding="cp1252", compression="zip")        # 8,760행 x 14열
#  받아지지 않으면: UCI 페이지에서 zip 을 내려받아 Colab 좌측 파일창에 올린 뒤
#  URL 대신 "seoul+bike+sharing+demand.zip" 을 넣으세요
raw.columns = ["date", "count", "hour", "temp", "humidity", "wind", "visibility",      # 위치 기반 리네임
               "dewpoint", "solar", "rain", "snow", "season", "holiday", "functioning"]
raw["datetime"] = (pd.to_datetime(raw["date"], format="%d/%m/%Y")    # 01/12/2017 = 2017년 12월 1일
                   + pd.to_timedelta(raw["hour"], unit="h"))
df = raw.sort_values("datetime").set_index("datetime")
df.index.freq = "h"                       # 등간격 1시간
_s = df["count"].astype(float).mask(df["functioning"].eq("No"))   # 휴무는 0 이 아니라 관측 없음
_y = _s.fillna(_s.shift(24 * 7))          # 1주일 전 같은 시각 값으로 채운다
df["demand"] = _y.fillna(_y.shift(-24 * 7))   # 1주일 전도 휴무인 시각만 1주일 뒤 값으로
hol_days = sorted(set(df.index[df["holiday"].eq("Holiday")].normalize()))
holidays_one = pd.DataFrame({"holiday": "공휴일", "ds": hol_days})   # 공휴일 18일을 이름 하나로 묶은 표

FIT_START = "2018-09-01 00:00"                                       # 학습은 원점마다 9월 1일 0시부터
ORIGINS = pd.date_range("2018-10-19 23:00", "2018-11-16 23:00", freq="7D")   # 금요일 23시 원점 5개
CAND_NAMES = ["셋째 날 SARIMA", "다섯째 날 Prophet 기본 구성"]
FC = {name: [] for name in CAND_NAMES}                               # 후보마다 원점별 168시간 예측
prev_params = None                                                   # 첫 원점은 기본 출발값에서 추정한다
for o in ORIGINS:
    fit_part = df.loc[FIT_START:o, "demand"]                          # 9월 1일부터 원점까지로 재적합
    week = pd.date_range(o + pd.Timedelta(hours=1), periods=168, freq="h")   # 원점 1시간 뒤부터 168시간
    sar = SARIMAX(fit_part, order=(1, 0, 1), seasonal_order=(1, 1, 1, 24)).fit(
        disp=False, start_params=prev_params)
    prev_params = sar.params                                         # 다음 원점은 이 계수에서 출발해 다시 추정한다
    FC["셋째 날 SARIMA"].append(pd.Series(np.asarray(sar.forecast(168), dtype=float), index=week))
    pro = Prophet(yearly_seasonality=False, weekly_seasonality=True, daily_seasonality=True,
                  holidays=holidays_one, uncertainty_samples=0)       # 구간 없이 예측값만 낸다
    pro.fit(pd.DataFrame({"ds": fit_part.index, "y": fit_part.values}))
    FC["다섯째 날 Prophet 기본 구성"].append(
        pd.Series(pro.predict(pd.DataFrame({"ds": week}))["yhat"].to_numpy(), index=week))
    print(f"  원점 {o:%m-%d %H}시 · 학습 {len(fit_part):,}시간으로 두 후보 적합 완료")

print(f"예측 원점 {len(ORIGINS)}개 · 후보 {len(CAND_NAMES)}개 : " + " · ".join(CAND_NAMES))
print("FC[이름][k] : k번째 원점 1시간 뒤부터 168시간의 예측 (인덱스가 그 168시각)")
print("쓸 수 있는 이름 : df (열 demand) · ORIGINS · CAND_NAMES · FC · rmse(a, b)")

  원점 10-19 23시 · 학습 1,176시간으로 두 후보 적합 완료
  원점 10-26 23시 · 학습 1,344시간으로 두 후보 적합 완료
  원점 11-02 23시 · 학습 1,512시간으로 두 후보 적합 완료
  원점 11-09 23시 · 학습 1,680시간으로 두 후보 적합 완료
  원점 11-16 23시 · 학습 1,848시간으로 두 후보 적합 완료
예측 원점 5개 · 후보 2개 : 셋째 날 SARIMA · 다섯째 날 Prophet 기본 구성
FC[이름][k] : k번째 원점 1시간 뒤부터 168시간의 예측 (인덱스가 그 168시각)
쓸 수 있는 이름 : df (열 demand) · ORIGINS · CAND_NAMES · FC · rmse(a, b)


In [3]:
# 미션 1 — 프롬프트   # ai: prompt
# PROMPT 를 쓰고 → 도우미에 "위 PROMPT 대로 코드를 써 줘" → 「이 셀에 넣기」 → ▶
PROMPT = """
먼저 지금 만들어진 셀이야. 

df 서울 공유자전거 시간 표. 열 demand 가 보정 수요(대)
ORIGINS · CAND_NAMES 예측 원점 5개(10월 19일～11월 16일의 금요일 23시)와 후보 이름 2개
FC 후보별 예측을 담은 사전. FC[이름][k] 는 k번째 원점 1시간 뒤부터 168시간의 예측이고, 인덱스가 그 168시각이다
rmse(a, b) 두 계열의 RMSE 를 구하는 함수(설치 셀)
그리고 아래를 만들어줘

채택 예측의 원점 5개 평균 RMSE : 단위는 '대'이고 소수 둘째자리까지, 원점마다 그 168시간의 실측 demand 와 예측 셋의 RMSE 를 구해야 해, 원점 5개를 평균해 가장 작은 예측의 값을 적고, 원점별 세 값과 가장 작은 예측 이름도 한 행씩 출력해줘
베이스라인보다 원점 5개 평균 RMSE가 작은 후보 수: 단위는 '개', 정수로
보고 한 행이 필요해 : 채택한 예측 이름과 채점 조건(예측 원점 5개 · 학습 9월 1일～원점 · 예측 기간 168시간 · 지표 RMSE).
다음은 그래프 한장을 출력해줘 : 원점 5개 평균 RMSE 막대 3개. 가로축 「예측」, 세로축 「원점 5개 평균 RMSE (대)」
"""
# --- 여기부터 AI 코드 ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ── 0. 예측 셋 준비: 후보 2개 + 베이스라인 ──
BASE_NAME = "계절 나이브 168"
NAIVE_168 = df["demand"].shift(168)          # 1주일 전 같은 시각 값
PRED_NAMES = list(CAND_NAMES) + [BASE_NAME]

# ── 1. 원점별 RMSE ──
rows = []
for k, origin in enumerate(ORIGINS):
    idx = FC[CAND_NAMES[0]][k].index          # 원점 1시간 뒤부터 168시각
    assert len(idx) == 168, f"원점 {k+1}의 예측 시각 수가 168이 아님"
    실측 = df.loc[idx, "demand"]

    row = {"원점": pd.Timestamp(origin).strftime("%m-%d %H시")}
    for name in CAND_NAMES:
        assert FC[name][k].index.equals(idx), f"{name} 원점 {k+1} 인덱스 불일치"
        row[name] = rmse(실측, FC[name][k])

    base_pred = NAIVE_168.loc[idx]
    assert base_pred.notna().all(), f"원점 {k+1} 베이스라인에 결측 있음"
    row[BASE_NAME] = rmse(실측, base_pred)

    row["가장 작은 예측"] = min(PRED_NAMES, key=lambda n: row[n])
    rows.append(row)

원점별 = pd.DataFrame(rows).set_index("원점")

print("■ 원점별 RMSE (대)")
for origin, r in 원점별.iterrows():
    값 = " · ".join(f"{n} {r[n]:,.2f}" for n in PRED_NAMES)
    print(f"  {origin} | {값} | 가장 작은 예측: {r['가장 작은 예측']}")

# ── 2. 원점 5개 평균 RMSE와 채택 예측 ──
평균 = 원점별[PRED_NAMES].mean().sort_values()
채택_이름 = 평균.idxmin()
채택_값 = 평균.min()

print("\n■ 원점 5개 평균 RMSE (대)")
for n, v in 평균.items():
    print(f"  {n}: {v:,.2f}대")
print(f"\n1. 채택 예측의 원점 5개 평균 RMSE: {채택_값:,.2f}대 ({채택_이름})")

# ── 3. 베이스라인보다 평균 RMSE가 작은 후보 수 ──
베이스_평균 = 평균[BASE_NAME]
이긴_후보수 = int(sum(평균[n] < 베이스_평균 for n in CAND_NAMES))
print(f"2. 베이스라인보다 원점 5개 평균 RMSE가 작은 후보 수: {이긴_후보수}개")

# ── 4. 보고 한 행 ──
print(f"3. 보고: 채택 예측 {채택_이름} "
      f"(채점 조건: 예측 원점 5개 · 학습 9월 1일～원점 · 예측 기간 168시간 · 지표 RMSE)")

# ── 5. 그래프: 원점 5개 평균 RMSE 막대 3개 ──
plot_order = list(CAND_NAMES) + [BASE_NAME]
values = [평균[n] for n in plot_order]
colors = ["#1f3b73" if n == 채택_이름 else "#a7c4e5" for n in plot_order]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(plot_order, values, color=colors)
ax.axhline(베이스_평균, color="gray", linestyle="--", linewidth=1,
           label=f"베이스라인 {베이스_평균:,.2f}대")
for b, v in zip(bars, values):
    ax.text(b.get_x() + b.get_width() / 2, v, f"{v:,.2f}",
            ha="center", va="bottom", fontsize=10)

ax.set_title("예측별 원점 5개 평균 RMSE (진한 색 = 채택)")
ax.set_xlabel("예측")
ax.set_ylabel("원점 5개 평균 RMSE (대)")
ax.set_ylim(0, max(values) * 1.15)
ax.legend()
plt.tight_layout()
plt.show()

■ 원점별 RMSE (대)
  10-19 23시 | 셋째 날 SARIMA 345.39 · 다섯째 날 Prophet 기본 구성 366.49 · 계절 나이브 168 306.52 | 가장 작은 예측: 계절 나이브 168
  10-26 23시 | 셋째 날 SARIMA 368.10 · 다섯째 날 Prophet 기본 구성 410.68 · 계절 나이브 168 418.18 | 가장 작은 예측: 셋째 날 SARIMA
  11-02 23시 | 셋째 날 SARIMA 508.05 · 다섯째 날 Prophet 기본 구성 474.90 · 계절 나이브 168 457.07 | 가장 작은 예측: 계절 나이브 168
  11-09 23시 | 셋째 날 SARIMA 317.68 · 다섯째 날 Prophet 기본 구성 342.82 · 계절 나이브 168 430.72 | 가장 작은 예측: 셋째 날 SARIMA
  11-16 23시 | 셋째 날 SARIMA 386.07 · 다섯째 날 Prophet 기본 구성 310.42 · 계절 나이브 168 205.25 | 가장 작은 예측: 계절 나이브 168

■ 원점 5개 평균 RMSE (대)
  계절 나이브 168: 363.55대
  다섯째 날 Prophet 기본 구성: 381.06대
  셋째 날 SARIMA: 385.06대

1. 채택 예측의 원점 5개 평균 RMSE: 363.55대 (계절 나이브 168)
2. 베이스라인보다 원점 5개 평균 RMSE가 작은 후보 수: 0개
3. 보고: 채택 예측 계절 나이브 168 (채점 조건: 예측 원점 5개 · 학습 9월 1일～원점 · 예측 기간 168시간 · 지표 RMSE)

──────────────────────────────────────────────────────
◼ 미션 1 채점 결과
✅ 채택 예측의 원점 5개 평균 RMSE — 출력에서 363.55 를 찾았습니다
✅ 베이스라인보다 원점 5개 평균 RMSE 가 작은 후보 수 — 출력에서 0 를 찾았습니다
✅ 그래프 — 기준대로 그렸습니다
🔓 정답이 열렸습니다 

## 미션 2 — 다음 주 운영 계획표

미션 1 에서 채택한 계절 나이브 168 이 계획표의 `예측` 열이 됩니다. 여기에 사람이 다시 볼 시각을 참으로 적는 `이상` 열을 더합니다. 미션 1 과 같은 순서로 합니다.

**이 미션의 질문**: 다음 주 168시각마다 몇 대를 내고, 어느 시각을 사람이 다시 볼지 한 장으로 어떻게 적을까요?

운영에서는 `이상` 열을 실측이 들어오는 대로 채웁니다. 이 실습에서는 11월 17～23일이 자료 안에서 이미 지난 주라 168시각의 실측이 `CNT` 에 있습니다. 계획표의 잔차는 `CNT` 에서 `예측` 열을 뺀 값이고, `RESID.loc[plan.index]` 와 같습니다.

잔차 강건 Z 는 0.6745 × (잔차 − 그 시각 중앙값) ÷ 그 시각 MAD 입니다. 예를 들어 8시는 기준 구간(10월 20일～11월 16일)의 중앙값이 −86대, MAD 가 142대입니다. 8시 잔차가 −370대라면 강건 Z 는 0.6745 × (−370 + 86) ÷ 142 = −1.35 입니다. 절댓값이 3.5 보다 작아 임계값 3.5 에서 이상으로 탐지되지 않습니다.

**내가 정할 것**: 하루 오탐 수 상한을 1건·2건·3건 가운데 하나로 정하고, 그 상한을 채우는 `ALERTS` 의 가장 작은 임계값을 고릅니다.

#### 만들 것

1. **`예측` 열**: 1주일 전 같은 시각의 보정 수요 `DEM.shift(168)` 에서 `plan` 의 168시각을 골라 `plan` 의 열로 둡니다
2. **임계값별 기준 구간 하루 오탐 수**: 단위 건 · 소수 둘째 자리. 기준 구간에서 라벨이 없는 시각의 강건 Z 절댓값이 임계값을 넘은 시각 수를 기준 구간의 날짜 수로 나눕니다. `ALERTS` 마다 「임계값 3.5 에서 기준 구간 하루 오탐 수 = 값 건」 형식으로 한 행씩 출력합니다
3. **`이상` 열**: 고른 임계값으로 168시각의 강건 Z 를 계산해, 절댓값이 임계값을 넘으면 참으로 적습니다. 고른 임계값은 `ALERT` 에 담습니다
4. **그래프 한 장**: 168시각의 강건 Z 선 하나와 고른 임계값의 ± 가로선. 가로축 「다음 주 시각」, 세로축 「잔차 강건 Z」. 선에는 3번에서 계산한 강건 Z 를 그대로 그립니다

**통과 기준**: 2번이 임계값마다 한 행씩 출력되어야 합니다. 3번 `이상` 열의 참인 시각이 `ALERT` 로 탐지한 시각과 같고, 4번 선의 값이 계획표 강건 Z 와 맞아야 통과입니다.

**쓸 수 있는 것**: 「미션 2 준비 셀」이 만들어 둔 이름입니다.

| 이름 | 무엇 |
|---|---|
| `plan` | 행이 다음 주 168시각(11월 17～23일)이고 아직 열이 없는 표 |
| `CNT` · `DEM` · `RESID` | 8,760시각의 원본 대여 수, 보정 수요, 잔차(원본에서 1주일 전 보정 수요를 뺀 값) |
| `HOUR` · `LABEL` · `IN_BASE` | 그 행의 시각 0～23, 정답 라벨(휴무·시간당 강수 10mm 이상), 기준 구간이면 참 |
| `MED_H` · `MAD_H` · `ALERTS` | 라벨 없는 기준 구간 잔차 599시각의 시각별 중앙값·MAD 24개씩, 임계값 (3.5, 4, 5) |

#### 프롬프트에 넣을 것(힌트)

- **두 강건 Z**: 기준 구간은 `RESID`, 계획표 168시각은 `CNT` 에서 `예측` 열을 뺀 잔차를 쓰고, 둘 다 그 시각의 `MED_H`·`MAD_H` 로 계산하라고 적습니다.
- **날짜 수**: 넘은 시각 수를 `IN_BASE` 의 시간 수 ÷ 24 로 나누라고 적습니다.
- **임계값**: 내가 정한 상한과, 그 상한을 채우는 가장 작은 임계값을 골라 `ALERT` 에 담으라는 말을 적습니다.

<img src="https://otexts.com/fpp3/fpp_files/figure-html/traintest-1.png" alt="오른쪽을 향한 시간축 위에 점이 한 줄로 놓인 그림. 왼쪽 파란 점 18개는 학습 자료, 오른쪽 주황 점 6개는 채점 자료다" width="640" height="64"/>
<p align="center"><em>▲ 계열 하나를 달력 순서대로 학습 구간과 채점 구간으로 나눈 그림. 계획표의 168행은 오른쪽 부분이고, 이상 탐지의 기준 구간은 왼쪽 부분의 마지막 4주다 (출처: fpp3 §5.8)</em></p>

#### 예상 적기

> **[예측] 하루 오탐 수를 1건 이하로 두려면 ALERTS 가운데 어느 임계값을 골라야 할까요?**
>
> 1. 3.5
> 2. 4
> 3. 5
>
> 답을 정한 뒤 아래를 읽어 보세요(정답 채점은 없습니다).

In [4]:
# 미션 2 · 준비 — 읽고 실행만 합니다. 이 셀이 미션 2 에 쓸 이름을 직접 만듭니다

_missing = [_n for _n in ("pd", "np", "plt", "datefmt") if _n not in globals()]
if _missing:                                                       # 맨 위 설치 셀을 아직 실행하지 않았습니다
    raise NameError("맨 위 설치 셀을 먼저 실행하세요 — 아직 없는 이름 "
                    + " · ".join(_missing))
if "judge" not in globals():                                       # 막지는 않고 알리기만 합니다
    print("ⓘ 맨 위 [채점] 셀을 아직 실행하지 않았습니다 — 그 셀을 한 번 실행해야"
          " 「PROMPT 셀」 아래에 채점 결과와 힌트가 나옵니다")

URL = "https://archive.ics.uci.edu/static/public/560/seoul+bike+sharing+demand.zip"
raw = pd.read_csv(URL, encoding="cp1252", compression="zip")
raw.columns = ["date", "count", "hour", "temp", "humidity", "wind", "visibility",
               "dewpoint", "solar", "rain", "snow", "season", "holiday", "functioning"]
raw["datetime"] = (pd.to_datetime(raw["date"], format="%d/%m/%Y")
                   + pd.to_timedelta(raw["hour"], unit="h"))
df = raw.sort_values("datetime").set_index("datetime")
df.index.freq = "h"
_s = df["count"].astype(float).mask(df["functioning"].eq("No"))   # 휴무 295시간은 결측으로 본다
_y = _s.fillna(_s.shift(24 * 7))
df["demand"] = _y.fillna(_y.shift(-24 * 7))                       # 1주일 전후 같은 시각으로 보정

TRAIN_END = "2018-11-16 23:00"                                    # 예측 원점: 이 시각까지의 실측만 씁니다
BASE_START = "2018-10-20 00:00"                                   # 기준 구간의 시작: 원점까지 마지막 4주
valid = df.loc["2018-11-17 00:00":"2018-11-23 23:00"]             # 다음 주 168시각 = 계획표의 행
plan = pd.DataFrame(index=valid.index)
plan.index.name = "시각"

# 여섯째 날 레퍼런스가 잔차의 실측으로 쓰는 원본 대여 수 count (계획표 168시각은 휴무가 없어 보정 수요와 같다)
CNT = df["count"].astype(float)
DEM = df["demand"].astype(float)                                  # 기댓값을 만들 보정 수요
HOUR = pd.Series(df.index.hour, index=df.index)
LABEL = df["functioning"].eq("No") | (df["rain"].astype(float) >= 10)   # 정답 라벨: 휴무 · 시간당 강수 10mm 이상
IN_BASE = pd.Series((df.index >= pd.Timestamp(BASE_START))
                    & (df.index <= pd.Timestamp(TRAIN_END)), index=df.index)
RESID = CNT - DEM.shift(168)                                       # 잔차 = 실측 − 1주일 전 같은 시각의 보정 수요
_base = pd.DataFrame({"r": RESID, "h": HOUR})[IN_BASE & ~LABEL & RESID.notna()]
_g = _base.groupby("h")["r"]
MED_H = _g.median()                                                # 기준 구간의 시각별 잔차 중앙값 24개
MAD_H = _g.apply(lambda s: float(np.median(np.abs(s - s.median()))))   # 기준 구간의 시각별 MAD 24개
ALERTS = (3.5, 4, 5)                                               # 비교할 임계값 3개

# 본문이 인용하는 수를 확인합니다
_n_base = int(IN_BASE.sum())
assert _n_base == 672 and len(_base) == 599 and len(plan) == 168
assert MAD_H[0] == 84 and MAD_H[4] == 13 and MED_H[8] == -86 and MAD_H[8] == 142
assert (CNT.loc[plan.index] == DEM.loc[plan.index]).all()
assert DEM.shift(168).loc[pd.Timestamp("2018-11-17 00:00")] == 754

print(f"기준 구간 : {BASE_START[:10]} 0시 ~ {TRAIN_END[:10]} 23시 · {_n_base}시간 · {_n_base // 24}일")
print(f"라벨 없는 기준 구간 잔차 : {len(_base)}시간 → 시각별 중앙값 MED_H · MAD MAD_H 24개씩")
print(f"계획표 plan : {plan.index[0]:%m-%d %H}시 ~ {plan.index[-1]:%m-%d %H}시 · 아직 열 없음")
print("쓸 수 있는 이름 : plan · CNT · DEM · RESID · HOUR · LABEL · IN_BASE"
      " · MED_H · MAD_H · ALERTS · datefmt")

기준 구간 : 2018-10-20 0시 ~ 2018-11-16 23시 · 672시간 · 28일
라벨 없는 기준 구간 잔차 : 599시간 → 시각별 중앙값 MED_H · MAD MAD_H 24개씩
계획표 plan : 11-17 00시 ~ 11-23 23시 · 아직 열 없음
쓸 수 있는 이름 : plan · CNT · DEM · RESID · HOUR · LABEL · IN_BASE · MED_H · MAD_H · ALERTS · datefmt


In [5]:
# 미션 2 — 프롬프트   # ai: prompt
# PROMPT 를 쓰고 → 도우미에 "위 PROMPT 대로 코드를 써 줘" → 「이 셀에 넣기」 → ▶
PROMPT = """
만들어야 할 것은 아래와 같아

예측 열 : 1주일 전 같은 시각의 보정 수요 DEM.shift(168) 에서 plan 의 168시각을 골라 plan 의 열로 나둬줘
임계값별 기준 구간 하루 오탐 수: 단위 건 · 소수 둘째 자리. 기준 구간에서 라벨이 없는 시각의 강건 Z 절댓값이 임계값을 넘은 시각 수를 기준 구간의 날짜 수로 나눠줘, ALERTS 마다 「임계값 3.5 에서 기준 구간 하루 오탐 수 = 값 건」 형식으로 한 행씩 출력해야 해.
이상 열 : 고른 임계값으로 168시각의 강건 Z 를 계산해, 절댓값이 임계값을 넘으면 참으로 적어줘, 고른 임계값은 ALERT 에 담아야 해.
다음은 그래프 한장이 필요해 : 168시각의 강건 Z 선 하나와 고른 임계값의 ± 가로선. 가로축 「다음 주 시각」, 세로축 「잔차 강건 Z」. 선에는 3번에서 계산한 강건 Z 를 그대로 그려줘

준비 된 셀은 아래와 같으니 또 만들지 말아줘
plan 행이 다음 주 168시각(11월 17～23일)이고 아직 열이 없는 표
CNT · DEM · RESID 8,760시각의 원본 대여 수, 보정 수요, 잔차(원본에서 1주일 전 보정 수요를 뺀 값)
HOUR · LABEL · IN_BASE 그 행의 시각 0～23, 정답 라벨(휴무·시간당 강수 10mm 이상), 기준 구간이면 참
MED_H · MAD_H · ALERTS 라벨 없는 기준 구간 잔차 599시각의 시각별 중앙값·MAD 24개씩, 임계값 (3.5, 4, 5)
"""
# --- 여기부터 AI 코드 ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ── 0. 내가 정하는 값: 하루 오탐 수 상한 (1, 2, 3 중 하나) ──
오탐_상한 = 2

# ── 1. 예측 열: 1주일 전 같은 시각의 보정 수요 ──
plan["예측"] = DEM.shift(168).loc[plan.index]
assert plan["예측"].notna().all(), "예측 열에 결측 있음"

# ── 강건 Z (8,760시각 전체) ──
Z = 0.6745 * (RESID - HOUR.map(MED_H)) / HOUR.map(MAD_H)

# ── 2. 임계값별 기준 구간 하루 오탐 수 ──
정상_기준 = IN_BASE.astype(bool) & ~LABEL.astype(bool)    # 기준 구간 + 라벨 없음
기준_일수 = IN_BASE.index[IN_BASE.astype(bool)].normalize().nunique()
print(f"기준 구간 날짜 수: {기준_일수}일 · 라벨 없는 기준 시각 수: {int(정상_기준.sum())}시각\n")

오탐표 = {}
for thr in sorted(ALERTS):
    오탐_시각수 = int((Z[정상_기준].abs() > thr).sum())
    하루_오탐 = 오탐_시각수 / 기준_일수
    오탐표[thr] = 하루_오탐
    print(f"임계값 {thr} 에서 기준 구간 하루 오탐 수 = {하루_오탐:.2f} 건")

# 상한을 채우는 가장 작은 임계값
통과 = [thr for thr in sorted(ALERTS) if 오탐표[thr] <= 오탐_상한]
assert 통과, f"하루 오탐 수 상한 {오탐_상한}건을 채우는 임계값이 ALERTS 안에 없음"
ALERT = 통과[0]
print(f"\n하루 오탐 수 상한 {오탐_상한}건 → 고른 임계값 ALERT = {ALERT} "
      f"(기준 구간 하루 오탐 수 {오탐표[ALERT]:.2f}건)")

# ── 3. 이상 열: 168시각의 강건 Z가 ±ALERT를 넘으면 참 ──
Z_PLAN = Z.loc[plan.index]
plan["이상"] = Z_PLAN.abs() > ALERT
print(f"다음 주 168시각 중 이상 시각 수: {int(plan['이상'].sum())}시각")
display(plan[["예측", "이상"]].head())

# ── 4. 그래프: 168시각의 강건 Z와 ±임계선 ──
fig, ax = plt.subplots(figsize=(13, 4.5))
ax.plot(plan.index, Z_PLAN, color="#1f3b73", linewidth=1.2, label="잔차 강건 Z")
ax.axhline(ALERT, color="#d9534f", linestyle="--", linewidth=1, label=f"임계값 ±{ALERT}")
ax.axhline(-ALERT, color="#d9534f", linestyle="--", linewidth=1)
ax.axhline(0, color="gray", linewidth=0.6)

ax.set_title(f"다음 주 168시각의 잔차 강건 Z (임계값 ±{ALERT}, 하루 오탐 상한 {오탐_상한}건)")
ax.set_xlabel("다음 주 시각")
ax.set_ylabel("잔차 강건 Z")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

기준 구간 날짜 수: 28일 · 라벨 없는 기준 시각 수: 599시각

임계값 3.5 에서 기준 구간 하루 오탐 수 = 2.43 건
임계값 4 에서 기준 구간 하루 오탐 수 = 1.64 건
임계값 5 에서 기준 구간 하루 오탐 수 = 0.75 건

하루 오탐 수 상한 2건 → 고른 임계값 ALERT = 4 (기준 구간 하루 오탐 수 1.64건)
다음 주 168시각 중 이상 시각 수: 1시각
                      예측   이상
시각                             
2018-11-17 00:00:00  754.0  False
2018-11-17 01:00:00  679.0  False
2018-11-17 02:00:00  568.0  False
2018-11-17 03:00:00  372.0  False
2018-11-17 04:00:00  223.0  False

──────────────────────────────────────────────────────
◼ 미션 2 채점 결과
✅ 임계값 3.5 에서 기준 구간 하루 오탐 수 — 출력에서 2.43 를 찾았습니다
✅ 임계값 5 에서 기준 구간 하루 오탐 수 — 출력에서 0.75 를 찾았습니다
✅ 그래프 — 기준대로 그렸습니다
✅ `이상` 열 — 고른 임계값 4 로 탐지한 시각 1개와 같습니다
🔓 정답이 열렸습니다 — 아래 「🔓 완성본 풀이 보기」를 펼쳐 내 코드와 견줘 보세요
🔓 미션 2 완성본. AI 코드 표시선 아래에 들어와야 할 코드는 이 정도입니다

■ 이 코드를 부른 프롬프트 예

    미션 2 준비 셀이 만든 plan, CNT, DEM, RESID, HOUR, LABEL, IN_BASE, MED_H, MAD_H, ALERTS 를 그대로 쓰고 다시 만들지 마.
    DEM.shift(168) 에서 plan 의 168시각을 골라 plan 의 "예측" 열을 만들어 줘.
    IN_BASE 가 참이고 LABEL 이 거짓이고 RESID 가 있는 시각만 골라 0.67

## 오늘의 결론

<div class="lms-summary">

**정리**

- 예측 원점 5개 평균 RMSE 는 계절 나이브 168 이 363.55대로 가장 작습니다. 셋째 날 SARIMA 는 약 385대, 다섯째 날 Prophet 기본 구성은 약 381대입니다.
- SARIMA 잔차는 시차 168 자기상관이 약 0.39 이고, 80% 예측구간을 벗어난 시각은 모두 하한 아래입니다. 그래서 구간은 참고로만 적습니다.
- 기준 구간 하루 오탐 수는 임계값 3.5·4·5 에서 2.43·1.64·0.75건입니다. 상한 1건이면 5 를 고르고, 11월 21일 0시·1시의 큰 빗나감도 이상으로 탐지되지 않습니다.
- 할 일: `예측` 열은 `DEM.shift(168)` 로 채워 금요일에 보냅니다. `이상` 열은 실측이 들어오는 대로 `MED_H`·`MAD_H` 와 고른 임계값으로 채웁니다.

다음 주 운영 예측은 계절 나이브 168 로 내고, 계획표의 `이상` 열은 기준 구간에서 고른 임계값으로 채웁니다.

</div>

> **[과제] 실습 제출 — 운영 예측 평가표와 다음 주 계획표**
>
> 미션 1·2 의 「PROMPT 셀」이 채점을 통과한 상태로 노트북을 제출합니다. PROMPT 는 내가 쓴 그대로 둡니다.
>
> 마감: 2026-10-15T23:59+09:00
>
> **평가 기준**
>
> | 기준 | 이렇게 하면 충족 | 배점 |
> | --- | --- | --- |
> | 미션 1 | 채점을 통과했고 PROMPT 에 만들 것이 내 말로 적혀 있다 | 50 |
> | 미션 2 | 채점을 통과했고 PROMPT 에 만들 것이 내 말로 적혀 있다 | 50 |
>
> 제출은 LMS 레슨 화면의 과제 카드에서 GitHub 링크로 합니다(이 파일에 적으면 제출되지 않습니다).

<div style="background:#F2F2F2;color:#000000;padding:28px;border-radius:12px;border:1px solid #E6E6E6;margin-top:16px;border-left:4px solid #051C2C">
<h2 style="color:#000000;margin:0 0 8px 0">수고했습니다 — 시계열 모듈 복습 완료</h2>
<div style="color:#656565">1～5일차 결정 규칙을 퀴즈 10개로 다시 확인하고, 예측 원점 5개로 고른 운영 예측과 이상 탐지 결과를 계획표 한 장에 담았습니다.</div>
</div>